# 🧬 Analog Propagation — a retrieval baseline for CASMI 2026

> **TL;DR** — MRR@25 here is an *exact-structure* metric, so generating SMILES is the wrong tool.
> This notebook **retrieves** from a 711,705-structure library and spends all its effort on
> *ordering* that list. The centrepiece is **mass-shifted analog propagation**, which identifies
> molecules that have **no reference spectrum anywhere** by matching them against their chemical
> relatives.

| Stage | What it adds | Public LB |
|---|---|---|
| Spectral library search only | finds Class-1 molecules | **0.151** |
| \+ analog propagation & calibrated ranker | reaches Class 2 | **0.233** |
| \+ leaderboard-calibrated class weighting | stops over-paying for Class 2 | **0.245** |
| \+ in-silico fragmentation channel | independent structural evidence | **0.266** |
| \+ spectrum→fingerprint model | chemistry predicted from the spectrum | **0.299** |
| \+ merged-spectrum model, seed-averaged ranker | matching train to test | **0.335** |
| \+ constants copied from a popular fork | **a regression** — one of them was a recall bug | **0.282** |
| \+ removing the candidate cap, 10 ppm window | restores 24% of Class-2 recall | **0.311** |
| \+ reverting every constant to its highest-scoring value | local validation preferred all of them; the leaderboard did not | **0.330** |
| \+ isolating the model channel (single-variable A/B) | m1@36k alone vs m1@24k+s2: **−0.019** | **0.311** |
| \+ a third model in the ensemble | ensembling *within* a view does not help | **0.329** |
| \+ **PubChem pool expansion**, top-50 isomers by `f·z` | the one lever validation cannot score | *this version* |
| \+ re-swept class prior and tree depth | retuned on the channel actually shipped | *this version* |

The two regression rows are left in on purpose; they are the most useful numbers in the table.

**0.282** — five constants were adopted from a well-scoring fork **as a bundle**. Four were fine;
one silently deleted a quarter of the reachable answers. The calibrated objective predicted the
drop at −0.057 before the leaderboard reported −0.053. See *The candidate cap*.

**0.311** — that fixed the cap but was still below 0.335, because three more changes had been
bundled in alongside it. Rather than guess, I spent a submission on a proper single-variable A/B:
two runs identical in every respect **except** the model channel. `m1@36k` merged-only scored
**0.311**; `m1@24k + s2` scored **0.330**. So the model channel carried the whole regression, the
analog constants were innocent, and — the useful part — **averaging a per-spectrum model with a
merged model is worth +0.019**, which is the largest single effect I have measured on this
leaderboard. Both local holdouts said the opposite. See *When validation stops predicting the
leaderboard*.

Four independent channels, one calibrated ranker. Every number in this notebook is measured — the
dead ends are reported alongside the wins, because knowing that PubChem *hurts* is worth as much as
knowing that analog propagation helps.

Everything is configurable from one `CFG` block. Each constant below is annotated with the
measurement that chose it, so you can see what is load-bearing and what is not.

---

## Why retrieval, and not generation

A prediction scores only if its **InChIKey14 matches exactly** after tautomer canonicalisation.
Getting a ~350 Da natural product exactly right by autoregressive decoding is vanishingly unlikely —
one misplaced hydroxyl and the score is zero. Ranking a finite list of *real* molecules turns an
impossible generation problem into a tractable ordering problem.

## The test set has three very different halves

The hosts split the ~400 scored molecules by novelty, and **hide which is which**:

| Class | Definition | The only thing that can find it |
|---|---|---|
| **1** | has public reference MS/MS | spectral library search |
| **2** | in PubChem/COCONUT, **no** reference spectra | database retrieval + ranking |
| **3** | not in PubChem at all | de novo |

The mix is hidden — so I measured it. A library-only submission of this engine scores **0.151**,
and the same engine scores **0.93 MRR** on a Class-1 simulation. That puts Class 1 at roughly
**0.151 / 0.93 ≈ 16%** of the test set.

**So ~84% of the test is out of reach of library search**, and that is where nearly all the
remaining score lives. Most public baselines spend all 25 slots on library hits ranked by cosine —
which is why the leaderboard clusters just above the Class-1 ceiling.


In [ ]:
TRAIN_SCRIPT_SRC = '"""Spectrum->fingerprint training with BCE + hard-negative contrastive ranking loss.\n   Ranking score is f_cand . z (exact Bayes log-likelihood up to a candidate-independent constant)."""\nimport numpy as np, torch, torch.nn as nn, torch.nn.functional as F, time, pickle, os, math, argparse\nimport fpmodel\n\ndef dev_of():\n    if torch.cuda.is_available(): return \'cuda\'\n    if torch.backends.mps.is_available(): return \'mps\'\n    return \'cpu\'\n\nclass Data:\n    def __init__(self, spec=\'ds/spec.npz\', fpdir=\'up_train\', pooldir=\'.\', device=\'cpu\', pool_on_gpu=True):\n        z=np.load(spec)\n        self.off=z[\'off\']; self.mz=z[\'mz\']; self.it=z[\'it\']; self.prec=z[\'prec\']\n        self.ad=z[\'ad\'].astype(np.int64); self.ins=z[\'ins\'].astype(np.int64)\n        self.ce=z[\'ce\']; self.mode=z[\'mode\']; self.six=z[\'six\']; self.is_val=z[\'is_val\']\n        m=pickle.load(open(f\'{fpdir}/fp_meta.pkl\',\'rb\')); self.nbits=m[\'nbits\']; self.skeys=m[\'keys\']\n        pm=pickle.load(open(f\'{pooldir}/pool_meta.pkl\',\'rb\'))\n        self.pmass=np.load(f\'{pooldir}/pool_mass.npy\')\n        PFP=np.unpackbits(np.load(f\'{pooldir}/pool_fp.npy\'),axis=1)[:, :self.nbits]\n        self.pkeys=pm[\'keys\']\n        k2p={k:i for i,k in enumerate(self.pkeys)}\n        self.s2p=np.array([k2p.get(k,-1) for k in self.skeys], dtype=np.int64)\n        self.device=device\n        if pool_on_gpu and device==\'cuda\':\n            self.PFP=torch.from_numpy(np.ascontiguousarray(PFP)).to(device)\n        else:\n            self.PFP=torch.from_numpy(np.ascontiguousarray(PFP))\n        # group spectra by structure so training inputs can be MERGED the way test molecules are:\n        # the hidden test gives 1-16 spectra per molecule (median 3), but training on single\n        # spectra creates a train/test mismatch.\n        order=np.argsort(self.six, kind=\'mergesort\')\n        s_sorted=self.six[order]\n        bounds=np.searchsorted(s_sorted, np.arange(s_sorted.max()+2))\n        self._grp_order=order; self._grp_bounds=bounds\n        self.tr=np.where(~self.is_val)[0]; self.va=np.where(self.is_val)[0]\n        keep = self.s2p[self.six[self.tr]]>=0\n        self.tr = self.tr[keep]\n        print(f\'nbits={self.nbits} pool={len(self.pmass)} train={len(self.tr)} val={len(self.va)}\',flush=True)\n\n    def sample_negs(self, pos_pidx, K, ppm=10.0, rng=None):\n        m=self.pmass[pos_pidx]; tol=m*ppm/1e6\n        lo=np.searchsorted(self.pmass, m-tol,\'left\'); hi=np.searchsorted(self.pmass, m+tol,\'right\')\n        out=np.empty((len(pos_pidx),K), dtype=np.int64)\n        for r,(a,b,p) in enumerate(zip(lo,hi,pos_pidx)):\n            n=b-a\n            if n<=1: out[r]=rng.integers(0,len(self.pmass),K)   # no isomers -> random decoys\n            else:\n                c=rng.integers(a,b,K)\n                bad=(c==p)\n                if bad.any(): c[bad]=np.where(c[bad]+1<b, c[bad]+1, a)\n                out[r]=c\n        return out\n\n    def peers(self, i, rng, kmax=4):\n        """Other spectra of the same structure (for merge augmentation)."""\n        s=self.six[i]\n        a,b=self._grp_bounds[s], self._grp_bounds[s+1]\n        if b-a<=1: return [i]\n        cand=self._grp_order[a:b]\n        k=int(rng.integers(2, min(kmax, b-a)+1))\n        pick=rng.choice(cand, size=min(k,len(cand)), replace=False)\n        if i not in pick: pick=np.concatenate([[i], pick[:-1]])\n        return list(pick)\n\n    def _merged(self, idxs, maxlen):\n        mzs=[]; its=[]\n        for j in idxs:\n            a,b=self.off[j], min(self.off[j]+maxlen, self.off[j+1])\n            mzs.append(self.mz[a:b]); its.append(self.it[a:b])\n        mz=np.concatenate(mzs); it=np.concatenate(its)\n        o=np.argsort(mz); mz,it=mz[o],it[o]\n        keep=np.ones(len(mz),bool)\n        for j in range(1,len(mz)):\n            if mz[j]-mz[j-1] < 0.005:\n                if it[j]>=it[j-1]: keep[j-1]=False\n                else: keep[j]=False\n        mz,it=mz[keep],it[keep]\n        if len(mz)>maxlen:\n            top=np.argsort(-it)[:maxlen]; top.sort(); mz,it=mz[top],it[top]\n        return mz,it\n\n    def batch(self, ids, K=0, rng=None, ppm=10.0, aug=False, merge_p=0.0):\n        B=len(ids); L=fpmodel.MAX_PEAKS\n        lens=np.minimum(self.off[ids+1]-self.off[ids], L); N=max(int(lens.max()),1)\n        mz=np.zeros((B,N),np.float32); it=np.zeros((B,N),np.float32); pad=np.ones((B,N),bool)\n        for r,(i,l) in enumerate(zip(ids,lens)):\n            a=self.off[i]; mz[r,:l]=self.mz[a:a+l]; it[r,:l]=self.it[a:a+l]; pad[r,:l]=False\n        if merge_p>0 and rng is not None:\n            for r,i in enumerate(ids):\n                if rng.random()>=merge_p: continue\n                pk=self.peers(i,rng)\n                if len(pk)<2: continue\n                m2,i2=self._merged(pk, N)\n                n2=len(m2)\n                mz[r,:]=0; it[r,:]=0; pad[r,:]=True\n                mz[r,:n2]=m2; it[r,:n2]=i2; pad[r,:n2]=False\n        if aug and rng is not None:\n            # spectra are noisy measurements; jitter them so the model cannot memorise\n            # exact peak patterns of the 276k training structures (the earlier run\n            # overfit hard: val hardneg-top1 peaked at 0.46 then decayed to 0.13).\n            keep = rng.random((B,N)) > rng.uniform(0.0, 0.30, size=(B,1))   # peak dropout\n            pad = pad | (~keep)\n            it = it * np.exp(rng.normal(0.0, 0.25, size=(B,N))).astype(np.float32)  # intensity jitter\n            mz = mz * (1.0 + rng.normal(0.0, 5e-6, size=(B,N))).astype(np.float32)  # m/z jitter (~5 ppm)\n            allpad = pad.all(1)\n            if allpad.any(): pad[allpad,0]=False\n        d=self.device; T=lambda x: torch.as_tensor(x,device=d)\n        ce=np.where(self.ce[ids]<0,25.0,self.ce[ids]).astype(np.float32)\n        inp=(T(mz),T(it),T(pad),T(self.prec[ids]),T(self.ad[ids]),T(self.ins[ids]),T(ce),T(self.mode[ids]))\n        pos=self.s2p[self.six[ids]]\n        cand=None\n        if K>0:\n            negs=self.sample_negs(pos,K,ppm,rng)\n            cand=np.concatenate([pos[:,None],negs],1)         # (B,K+1), col 0 = positive\n            cand=torch.as_tensor(cand, device=self.PFP.device)\n        ypos=self.PFP[torch.as_tensor(pos, device=self.PFP.device)].to(d).float()\n        return inp, ypos, cand\n\ndef main():\n    ap=argparse.ArgumentParser()\n    ap.add_argument(\'--steps\',type=int,default=80000); ap.add_argument(\'--bs\',type=int,default=256)\n    ap.add_argument(\'--lr\',type=float,default=3e-4); ap.add_argument(\'--d\',type=int,default=512)\n    ap.add_argument(\'--layers\',type=int,default=6); ap.add_argument(\'--K\',type=int,default=63)\n    ap.add_argument(\'--lam\',type=float,default=1.0); ap.add_argument(\'--warm\',type=int,default=2000)\n    ap.add_argument(\'--out\',default=\'fp_model.pt\'); ap.add_argument(\'--resume\',default=\'\')\n    ap.add_argument(\'--val_every\',type=int,default=2000); ap.add_argument(\'--max_minutes\',type=float,default=1e9)\n    ap.add_argument(\'--seed\',type=int,default=7)\n    ap.add_argument(\'--merge_p\',type=float,default=0.6)\n    ap.add_argument(\'--spec\',default=\'ds/spec.npz\'); ap.add_argument(\'--fpdir\',default=\'up_train\'); ap.add_argument(\'--pooldir\',default=\'.\')\n    a=ap.parse_args()\n    dev=dev_of(); print(\'device\',dev,flush=True)\n    D=Data(a.spec,a.fpdir,a.pooldir,device=dev)\n    model=fpmodel.FPNet(D.nbits,d=a.d,layers=a.layers).to(dev)\n    print(\'params %.1fM\'%(sum(p.numel() for p in model.parameters())/1e6),flush=True)\n    torch.manual_seed(a.seed)\n    opt=torch.optim.AdamW(model.parameters(),lr=a.lr,weight_decay=0.01,betas=(0.9,0.98))\n    scaler=torch.amp.GradScaler(\'cuda\') if dev==\'cuda\' else None\n    start=0\n    if a.resume and os.path.exists(a.resume):\n        ck=torch.load(a.resume,map_location=dev); model.load_state_dict(ck[\'model\'])\n        try: opt.load_state_dict(ck[\'opt\'])\n        except Exception: pass\n        start=ck[\'step\']; print(\'resume\',start,flush=True)\n    def lr_at(s):\n        if s<a.warm: return a.lr*s/max(1,a.warm)\n        p=(s-a.warm)/max(1,a.steps-a.warm); return a.lr*(0.02+0.98*0.5*(1+math.cos(math.pi*min(p,1.0))))\n    rng=np.random.default_rng(a.seed)\n    t0=time.time(); rb=rc=racc=0.0; nr=0\n    best_val=-1.0   # keep the checkpoint that generalises best, not the last one\n    def save(step):\n        torch.save({\'model\':model.state_dict(),\'opt\':opt.state_dict(),\'step\':step,\n                    \'nbits\':D.nbits,\'d\':a.d,\'layers\':a.layers}, a.out)\n    for step in range(start,a.steps):\n        for g in opt.param_groups: g[\'lr\']=lr_at(step)\n        ids=rng.choice(D.tr,size=a.bs,replace=False)\n        inp,ypos,cand = D.batch(ids,K=a.K,rng=rng,aug=True,merge_p=a.merge_p)\n        opt.zero_grad(set_to_none=True)\n        ctx = torch.autocast(\'cuda\',dtype=torch.float16) if dev==\'cuda\' else torch.autocast(\'mps\',dtype=torch.float16) if dev==\'mps\' else torch.autocast(\'cpu\',enabled=False)\n        with ctx:\n            z=model(*inp)\n        z=z.float()\n        lb=F.binary_cross_entropy_with_logits(z,ypos)\n        FPc=D.PFP[cand].to(z.device).float()                     # (B,K+1,nbits)\n        raw=torch.bmm(FPc, z.unsqueeze(-1)).squeeze(-1)          # (B,K+1) = f.z  (exact Bayes LL up to const)\n        # centre within the example: subtracting (mean_c f_c).z is a per-example constant,\n        # so the ranking is untouched, but the magnitudes stay small enough for a stable softmax.\n        sc=(raw - raw.mean(dim=1, keepdim=True)) / math.sqrt(D.nbits)\n        tgt=torch.zeros(len(ids),dtype=torch.long,device=z.device)\n        lc=F.cross_entropy(sc,tgt)\n        loss=lb+a.lam*lc\n        if not torch.isfinite(loss):\n            print(f\'  non-finite loss at step {step}; skipping\', flush=True)\n            opt.zero_grad(set_to_none=True); continue\n        if scaler is not None:\n            scaler.scale(loss).backward(); scaler.unscale_(opt)\n            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); scaler.step(opt); scaler.update()\n        else:\n            loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0); opt.step()\n        rb+=lb.item(); rc+=lc.item(); racc+=(sc.argmax(1)==0).float().mean().item(); nr+=1\n        if step%200==0:\n            el=time.time()-t0\n            print(f\'step {step} bce {rb/max(nr,1):.4f} ctr {rc/max(nr,1):.4f} top1 {racc/max(nr,1):.3f} lr {lr_at(step):.2e} {el:.0f}s {(step-start+1)*a.bs/max(el,1):.0f} spec/s\',flush=True)\n            rb=rc=racc=0.0; nr=0\n        if (step+1)%a.val_every==0:\n            model.eval(); vb=[];vacc=[]\n            with torch.no_grad():\n                vids=rng.choice(D.va,size=min(2048,len(D.va)),replace=False)\n                for s in range(0,len(vids),128):\n                    ii=vids[s:s+128]\n                    pos=D.s2p[D.six[ii]]\n                    if (pos<0).any(): ii=ii[pos>=0]\n                    if len(ii)==0: continue\n                    inp,ypos,cand=D.batch(ii,K=a.K,rng=rng,merge_p=a.merge_p)\n                    z=model(*inp).float()\n                    vb.append(F.binary_cross_entropy_with_logits(z,ypos).item())\n                    FPc=D.PFP[cand].to(z.device).float()\n                    raw=torch.bmm(FPc,z.unsqueeze(-1)).squeeze(-1)\n                    sc=(raw-raw.mean(dim=1,keepdim=True))/math.sqrt(D.nbits)\n                    vacc.append((sc.argmax(1)==0).float().mean().item())\n            va=float(np.mean(vacc))\n            print(f\'  VAL step {step} bce {np.mean(vb):.4f} hardneg-top1 {va:.3f}\'\n                  + (\'  <- best\' if va>best_val else \'\'),flush=True)\n            model.train()\n            if va>best_val:\n                best_val=va; save(step+1)          # only overwrite when validation improves\n        if (time.time()-t0)/60>a.max_minutes:\n            print(\'time budget reached; keeping best-val checkpoint\',flush=True); break\n    print(\'done. best hardneg-top1 = %.3f\'%best_val,flush=True)\n\nif __name__==\'__main__\': main()\n'


In [ ]:

# ===================================================================================
#  CONFIG — every knob in one place. Each is annotated with the measurement behind it.
# ===================================================================================
class CFG:
    # --- candidate generation -------------------------------------------------------
    PPM_WIN      = 10.0   # neutral-mass window for candidates. Tighter IS better -- up to the
                          # point where it starts deleting answers, which is the same trap as
                          # CAND_CAP below. Check window RECALL, not just MRR:
                          #   5.0 ppm 0.972 | 7.0 0.992 | 8.5 0.992 | 10.0 1.000 | 12+ 1.000
                          # 10 ppm is the smallest window that loses nothing, and everything
                          # wider only adds decoys. Full four-channel ranker agrees:
                          #   8.5 -> predLB 0.361 | 10.0 -> 0.371 | 12.0 -> 0.363
                          # (Much wider genuinely does hurt: +-20 -> 0.509, +-30 -> 0.500 C2 MRR.)
                          # timsTOF precursor error stays under ~9 ppm (+1.4 ppm offset).
    PPM_FALLBACK = 30.0   # only used if the tight window returns nothing at all.

    # --- spectrum cleaning ----------------------------------------------------------
    INT_FLOOR    = 0.002  # drop peaks below this fraction of the base peak
    MAX_PEAKS    = 256    # keep the N most intense peaks after the floor
    MZ_TOL       = 0.01   # Da tolerance when matching two peaks
    INT_POWER    = 1.0    # intensity transform before similarity (1.0 + entropy weighting
                          # beat sqrt: Class-1 0.919 vs 0.895)
    ENT_WEIGHT   = True   # Li et al. 2021 entropy weighting of low-entropy spectra

    # --- analog propagation (the main idea) -----------------------------------------
    ANALOG_WIN   = 200.0  # +- Da mass-shift window. +-400 gave no gain (0.520 vs 0.521).
    N_ANALOG     = 200     # analogs kept per molecule. Flat above 80 -- predicted LB 0.3683 (60),
                          # 0.3709 (80), 0.3705 (100), 0.3703 (140). Nothing to win here.
    SIM_POWER    = 3.0    # sim^p weighting. p=1 -> 0.498, p=3 -> 0.521, p=4 -> 0.525 on local
                          # validation -- but see the model-selection section: that validation set
                          # is the one the checkpoint was early-stopped on, so small local wins on
                          # it are not trustworthy. p=3 is the value that actually scored 0.335.

    # --- ranker ---------------------------------------------------------------------
    W1_PRIORS    = (0.30, 0.60)  # REVERTED. A narrow plateau (.40,.45,.50) scored 0.3789 in my
                          # own sweep and then LOST on the leaderboard (0.335 -> 0.330). The sweep
                          # was in-sample: it trained on all 819 query groups and evaluated on 250
                          # of them. See the ranker section -- hold out BY QUERY or you are just
                          # measuring capacity to memorise your own evaluation set.
    SEEDS        = (0, 1, 2, 3)  # seed alone moves the LB by ~0.006; bag several.
    W1           = 0.50   # weight on the Class-1 simulation. NOT the class share (0.16) --
                          # it is the leaderboard-calibrated value, chosen by 5-fold CV held out
                          # by query in make_ranker3.py.
    GBM = dict(max_depth=6, max_iter=500, learning_rate=0.03,
               min_samples_leaf=80, l2_regularization=1.0)
               # Depth 6, also REVERTED from 10. In-sample, depth 10 looked worth +0.007; on the
               # leaderboard it was not. Deeper trees fit the evaluation queries better precisely
               # because those queries were in the training rows.
    USE_BIO_DB   = False  # add ChEBI + LIPID MAPS. Costs -0.026 Class-2 MRR in dilution but
                          # adds 7-19% coverage of in-library structures. Looked net-positive
                          # on validation but the leaderboard disagreed (0.299 -> 0.295), so it
                          # ships OFF. Flip it if your pool recall differs.
                          # (Adding all of PubChem instead costs -0.35: measured, do not.)
    CAND_CAP     = 500    # pure runtime guard on in-silico fragmentation (~14 ms/candidate).
                          # It is deliberately LARGE. A cap of 80 ranked by "library hit, then
                          # closest in mass" looks like an adaptive version of "tighter windows
                          # win" -- it is not, it is a recall bug. Class-2 answers have
                          # library_sim = 0 *by definition* (no reference spectrum exists), so
                          # they get ordered by mass alone, which inside a +-8.5 ppm window is
                          # arbitrary. Measured truth retention on the Class-2 holdout:
                          #   no cap 0.992 | cap 400 0.992 | cap 200 0.952 | cap 80 0.752
                          # i.e. a cap of 80 throws away a QUARTER of the reachable answers.
                          # Class 1 is untouched (0.992 at every cap) because lv*100 protects it,
                          # which is exactly why the bug survives casual validation.
                          # Median window is 52 candidates, max 401, so 500 essentially never
                          # fires -- and when it does it ranks by the model, not by mass.
    PC_TOPK      = 50     # PubChem isomers admitted per query, ranked by the model's f.z.
                          # 0 disables the expansion entirely. Admitting ALL isomers costs a
                          # dilution factor d ~ 0.52; admitting the model's top 50 keeps d close
                          # to 1 while still reaching answers COCONUT does not contain.
    PC_WINCAP    = 10000  # isomers fingerprinted per query. This is the binding constraint on
                          # the whole expansion, not the admission cut: retain(50) = presence x
                          # conditional-retention = 0.788 x 0.79, and presence is set entirely by
                          # this number. Measured presence: 2k -> 0.788 | 5k -> 0.884 |
                          # 12k -> 0.940 | all -> 0.948. Windows run to 36,518 isomers.
                          # Costs ~5x the fingerprinting time; widen it before tuning PC_TOPK.
    TOPN         = 25     # the metric allows 25 guesses; there is no penalty for using them all


In [ ]:

import os, glob, time, pickle, math
import numpy as np, pandas as pd, pyarrow.parquet as pq, pyarrow as pa
T0 = time.time()

def find(name):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if not hits: raise FileNotFoundError(name)
    return sorted(hits, key=len)[0]

COMP   = os.path.dirname(find('test.parquet'))
TRAIN  = os.path.join(COMP, 'train.parquet')
TEST   = os.path.join(COMP, 'test.parquet')
SAMPLE = os.path.join(COMP, 'sample_submission.csv')
print('competition files:', os.listdir(COMP))


In [ ]:

# RDKit is not in the Kaggle image and internet is off for code competitions, so install the
# wheel from an attached dataset. Only the fragmentation channel needs it.
import subprocess, sys, glob
whl = glob.glob('/kaggle/input/**/rdkit-*.whl', recursive=True)
if whl:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-index', whl[0]], check=False)
try:
    from rdkit import Chem
    HAVE_RDKIT = True
except Exception:
    HAVE_RDKIT = False
print('RDKit available:', HAVE_RDKIT, '(fragmentation channel is optional - the notebook runs without it)')


---

## 🔑 The main idea: mass-shifted analog propagation

A Class-2 molecule has **no reference spectrum**. But its **structural relatives usually do**.

Two molecules that share a scaffold and differ by one substituent fragment into *largely the same
ions, offset by a constant mass*. So a **modified (mass-shifted) similarity** still matches a
spectrum against relatives of molecules the library has never seen. Propagating those relatives'
known structures onto candidates gives:

$$\mathrm{score}(c)\;=\;\max_{a\,\in\,\text{analogs}}\;\mathrm{sim}(a)^{p}\cdot\mathrm{Tanimoto}(f_c,\,f_a)$$

**Measured on a held-out Class-2 simulation (250 natural products, all their spectra removed from
the library):**

| Ranking of the same candidates | MRR@25 |
|---|---|
| random order | 0.138 |
| by mass error *(what public baselines do)* | 0.164 |
| NP-likeness prior | 0.037 |
| **analog propagation** | **0.521** |

It needs no Class-1 branch either: for a Class-1 molecule the library contains the molecule
*itself*, which shows up as an analog at zero mass shift with Tanimoto 1.0 and is ranked first
automatically.

It also generalises. Repeating the experiment on **569 obscure** held-out structures (rather than
common natural products) gives **0.516** — statistically identical, so this is not an artefact of
well-studied compounds.


In [ ]:
"""Similarity kernels: weighted cosine + spectral entropy similarity (Li et al. 2021)."""
import numpy as np
from numba import njit, prange

@njit(cache=True, fastmath=True)
def _clean(mz, it, floor, topk, power, ent_weight):
    n=len(mz)
    if n==0: return np.empty(0,np.float32), np.empty(0,np.float32)
    mx=0.0
    for i in range(n):
        if it[i]>mx: mx=it[i]
    if mx<=0: return np.empty(0,np.float32), np.empty(0,np.float32)
    thr=floor*mx; c=0
    for i in range(n):
        if it[i]>=thr: c+=1
    idx=np.empty(c,np.int64); j=0
    for i in range(n):
        if it[i]>=thr: idx[j]=i; j+=1
    if c>topk:
        v=np.empty(c,np.float32)
        for i in range(c): v[i]=it[idx[i]]
        o=np.argsort(v)[c-topk:]
        k2=np.empty(topk,np.int64)
        for i in range(topk): k2[i]=idx[o[i]]
        k2.sort(); idx=k2; c=topk
    om=np.empty(c,np.float32); oi=np.empty(c,np.float32)
    s=0.0
    for i in range(c):
        om[i]=mz[idx[i]]; v=it[idx[i]]**power; oi[i]=v; s+=v
    if s>0:
        for i in range(c): oi[i]/=s
    if ent_weight:
        S=0.0
        for i in range(c):
            if oi[i]>0: S-=oi[i]*np.log(oi[i])
        if S<3.0:
            w=0.25+0.25*S; s2=0.0
            for i in range(c): oi[i]=oi[i]**w; s2+=oi[i]
            if s2>0:
                for i in range(c): oi[i]/=s2
    return om, oi

@njit(cache=True, fastmath=True)
def entropy_sim(qmz,qp,cmz,cp,tol):
    i=0;j=0;n=len(qmz);m=len(cmz)
    SA=0.0
    for x in range(n):
        if qp[x]>0: SA-=qp[x]*np.log(qp[x])
    SB=0.0
    for x in range(m):
        if cp[x]>0: SB-=cp[x]*np.log(cp[x])
    SAB=0.0; tot=0.0
    buf=np.empty(n+m,np.float64); b=0
    while i<n and j<m:
        d=qmz[i]-cmz[j]
        if d<-tol: buf[b]=qp[i]; i+=1; b+=1
        elif d>tol: buf[b]=cp[j]; j+=1; b+=1
        else: buf[b]=qp[i]+cp[j]; i+=1; j+=1; b+=1
    while i<n: buf[b]=qp[i]; i+=1; b+=1
    while j<m: buf[b]=cp[j]; j+=1; b+=1
    for x in range(b): tot+=buf[x]
    if tot<=0: return 0.0
    for x in range(b):
        v=buf[x]/tot
        if v>0: SAB-=v*np.log(v)
    return 1.0-(2.0*SAB-SA-SB)/np.log(4.0)

@njit(cache=True, fastmath=True)
def cos_sim(qmz,qp,cmz,cp,tol):
    i=0;j=0;n=len(qmz);m=len(cmz); dot=0.0; na=0.0; nb=0.0
    for x in range(n): na+=qp[x]*qp[x]
    for x in range(m): nb+=cp[x]*cp[x]
    while i<n and j<m:
        d=qmz[i]-cmz[j]
        if d<-tol: i+=1
        elif d>tol: j+=1
        else: dot+=qp[i]*cp[j]; i+=1; j+=1
    if na<=0 or nb<=0: return 0.0
    return dot/np.sqrt(na*nb)

@njit(cache=True, fastmath=True, parallel=True)
def search(qmz,qp,cand,off,allmz,allin,tol,floor,topk,power,ent_weight,kind):
    out=np.zeros(len(cand),np.float32)
    for k in prange(len(cand)):
        c=cand[k]; a=off[c]; b=off[c+1]
        if b<=a: continue
        cm,cp=_clean(allmz[a:b],allin[a:b],floor,topk,power,ent_weight)
        if len(cm)==0: continue
        out[k]= entropy_sim(qmz,qp,cm,cp,tol) if kind==1 else cos_sim(qmz,qp,cm,cp,tol)
    return out

def prep(mz,it,floor=0.002,topk=256,power=0.5,ent_weight=False):
    return _clean(np.asarray(mz,np.float32),np.asarray(it,np.float32),floor,topk,power,ent_weight)

@njit(cache=True, fastmath=True)
def entropy_sim_shift(qmz,qp,cmz,cp,tol,shift):
    """Best of direct and mass-shifted entropy similarity."""
    a = entropy_sim(qmz,qp,cmz,cp,tol)
    if shift > -0.001 and shift < 0.001: return a
    sm = np.empty(len(cmz), np.float32)
    for i in range(len(cmz)): sm[i]=cmz[i]+shift
    b = entropy_sim(qmz,qp,sm,cp,tol)
    return a if a>b else b

@njit(cache=True, fastmath=True, parallel=True)
def search_shift(qmz,qp,cand,off,allmz,allin,tol,floor,topk,power,ent_weight,kind,shift):
    out=np.zeros(len(cand),np.float32)
    for k in prange(len(cand)):
        c=cand[k]; a=off[c]; b=off[c+1]
        if b<=a: continue
        cm,cp=_clean(allmz[a:b],allin[a:b],floor,topk,power,ent_weight)
        if len(cm)==0: continue
        out[k]=entropy_sim_shift(qmz,qp,cm,cp,tol,shift[k])
    return out


In [ ]:

# ===================================================================================
#  Adduct -> neutral mass.  The instrument measures the *ion*; candidates are neutral
#  molecules, so every adduct has to be undone before we can compare masses.
# ===================================================================================

MASS = dict(C=12.0,H=1.00782503207,N=14.0030740048,O=15.9949146196,P=30.97376163,
            S=31.97207100,F=18.99840322,Cl=34.96885268,Br=78.9183371,I=126.904473,
            Na=22.9897692809,K=38.96370668,Si=27.9769265325,B=11.0093054,Se=79.9165213)
E=0.00054857990; PROTON=MASS['H']-E; H2O=2*MASS['H']+MASS['O']
NH4=MASS['N']+4*MASS['H']; FORMATE=MASS['C']+2*MASS['H']+2*MASS['O']
ACETATE=2*MASS['C']+4*MASS['H']+2*MASS['O']
ADDUCTS = {
 "[M+H]+":(1,1,PROTON), "[M+NH4]+":(1,1,NH4-E), "[M+Na]+":(1,1,MASS['Na']-E),
 "[M+K]+":(1,1,MASS['K']-E), "[M-H2O+H]+":(1,1,PROTON-H2O), "[M-2H2O+H]+":(1,1,PROTON-2*H2O),
 "[M+2H]2+":(1,2,2*PROTON), "[M]+":(1,1,-E), "[M-H2O]+":(1,1,-E-H2O),
 "[M+CH3OH+H]+":(1,1,PROTON+MASS['C']+4*MASS['H']+MASS['O']),
 "[M+CH3CN+H]+":(1,1,PROTON+2*MASS['C']+3*MASS['H']+MASS['N']),
 "[M-H]-":(1,1,-PROTON), "[M-H2O-H]-":(1,1,-PROTON-H2O), "[M+CH2O2-H]-":(1,1,FORMATE-PROTON),
 "[M+C2H4O2-H]-":(1,1,ACETATE-PROTON), "[M+Cl]-":(1,1,MASS['Cl']+E), "[M]-":(1,1,E),
 "[M-2H]-":(1,2,-2*PROTON), "[M+Na-2H]-":(1,1,MASS['Na']-2*PROTON),
 "[2M+H]+":(2,1,PROTON), "[2M+Na]+":(2,1,MASS['Na']-E), "[2M+NH4]+":(2,1,NH4-E),
 "[2M+K]+":(2,1,MASS['K']-E), "[2M-H]-":(2,1,-PROTON), "[2M+CH2O2-H]-":(2,1,FORMATE-PROTON),
 "[2M+C2H4O2-H]-":(2,1,ACETATE-PROTON), "[2M+Na-2H]-":(2,1,MASS['Na']-2*PROTON),
 "[3M+H]+":(3,1,PROTON), "[3M-H]-":(3,1,-PROTON),
}
def neutral_mass(mz, adduct):
    out=np.full(len(mz), np.nan); ad=np.asarray(adduct, dtype=object)
    for a,(n,z,d) in ADDUCTS.items():
        m=(ad==a)
        if m.any(): out[m]=(mz[m]*z-d)/n
    return out


# ===================================================================================
#  Library + candidate pool
# ===================================================================================
def load_library(path):
    t0 = time.time()
    t = pq.read_table(path, columns=['inchikey14','normalized_smiles','adduct','precursor_mz',
                                     'ms2_mzs','ms2_normalized_intensities'])
    mzc = t.column('ms2_mzs').combine_chunks(); itc = t.column('ms2_normalized_intensities').combine_chunks()
    off = mzc.offsets.to_numpy().astype(np.int64)
    allmz = mzc.values.to_numpy(zero_copy_only=False).astype(np.float32)
    allin = itc.values.to_numpy(zero_copy_only=False).astype(np.float32)
    prec = t.column('precursor_mz').to_numpy(zero_copy_only=False).astype(np.float64)
    add = np.asarray(t.column('adduct').cast(pa.string()).to_pylist(), dtype=object)
    ik  = np.asarray(t.column('inchikey14').cast(pa.string()).to_pylist(), dtype=object)
    smi = np.asarray(t.column('normalized_smiles').cast(pa.string()).to_pylist(), dtype=object)
    nm  = neutral_mass(prec, add); ok = np.isfinite(nm)
    order = np.argsort(np.where(ok, nm, 1e18), kind='mergesort')
    best = {}
    for k, s in zip(ik, smi):
        if k and s and k not in best: best[k] = s
    print(f'library: {len(off)-1:,} spectra / {len(best):,} structures  ({time.time()-t0:.0f}s)', flush=True)
    return dict(off=off, mz=allmz, it=allin, nm=nm, ik=ik, best=best,
                order=order, snm=nm[order], n_ok=int(ok.sum()))

def lib_window(L, target, tol):
    lo = np.searchsorted(L['snm'][:L['n_ok']], target-tol, 'left')
    hi = np.searchsorted(L['snm'][:L['n_ok']], target+tol, 'right')
    return L['order'][lo:hi]

def build_rep(L):
    """One representative spectrum per structure (the richest), sorted by neutral mass.
       Using 3 per structure was WORSE (0.49 vs 0.52): extra spectra raise the max similarity
       of irrelevant structures too, which flattens the discrimination."""
    npk = np.diff(L['off']); best = {}; ik = L['ik']
    for i in range(len(ik)):
        k = ik[i]
        if k and (k not in best or npk[i] > npk[best[k]]): best[k] = i
    rep = np.array(sorted(best.values()))
    nm = L['nm'][rep]; ok = np.isfinite(nm)
    rep = rep[ok]; nm = nm[ok]; key = ik[rep]
    o = np.argsort(nm)
    return rep[o], key[o], nm[o]

# ===================================================================================
#  The two evidence channels
# ===================================================================================
def clean(mz, it):
    return _clean(np.asarray(mz, np.float32), np.asarray(it, np.float32),
                  CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT)

def lib_sim(L, specs, target):
    """CLASS 1: direct match against library spectra of the same neutral mass."""
    cand = lib_window(L, target, target*CFG.PPM_WIN/1e6)
    if len(cand) == 0: return {}
    agg = {}
    for mz, it in specs:
        qm, qp = clean(mz, it)
        if len(qm) == 0: continue
        sc = search(qm, qp, cand, L['off'], L['mz'], L['it'],
                    CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS, CFG.INT_POWER, CFG.ENT_WEIGHT, 1)
        for c, s in zip(cand, sc):
            k = L['ik'][c]
            if s > agg.get(k, -1.0): agg[k] = float(s)
    return agg

def analog_sim(L, specs, target, rep, rep_key, rep_nm):
    """CLASS 2: mass-SHIFTED match over a wide window. Relatives of the unknown fragment
       into the same ions offset by the mass difference, so they still match."""
    lo = np.searchsorted(rep_nm, target-CFG.ANALOG_WIN, 'left')
    hi = np.searchsorted(rep_nm, target+CFG.ANALOG_WIN, 'right')
    cand = rep[lo:hi]
    if len(cand) == 0: return []
    shift = (target - rep_nm[lo:hi]).astype(np.float32)
    ckey = rep_key[lo:hi]; agg = {}
    for mz, it in specs:
        qm, qp = clean(mz, it)
        if len(qm) == 0: continue
        sc = search_shift(qm, qp, cand, L['off'], L['mz'], L['it'],
                          CFG.MZ_TOL, CFG.INT_FLOOR, CFG.MAX_PEAKS,
                          CFG.INT_POWER, CFG.ENT_WEIGHT, 1, shift)
        for c, k, s in zip(cand, ckey, sc):
            if s > agg.get(k, -1.0): agg[k] = float(s)
    return sorted(agg.items(), key=lambda x: -x[1])[:CFG.N_ANALOG]


---

## The candidate pool, and why it is *small on purpose*

Candidates = **training-set structures ∪ COCONUT**, deduplicated on InChIKey14 →
**711,705** structures with precomputed fingerprints. COCONUT covers **99.6%** of
`enveda-np-examples`, the library the hosts describe as closest to the test set.

A ±10 ppm neutral-mass window gives a **median of 56 candidates**, and contained the true structure
**100%** of the time in validation (timsTOF precursor error stays under ~9 ppm, with a systematic
+1.4 ppm calibration offset).

**Bigger is worse — this is measured, not assumed:**

| Candidate window | median candidates | Class-2 MRR |
|---|---|---|
| **±10 ppm** | **56** | **0.521** |
| ±20 ppm | 76 | 0.509 |
| ±30 ppm | 105 | 0.500 |
| \+ all PubChem isomers (~3,900 more) | ~4,000 | **0.350** |
| \+ only the *top 10* PubChem isomers | 66 | **0.337** |

Adding PubChem is catastrophic, and note that admitting only its **top 10** is just as bad. The
reason is instructive: analog-Tanimoto is maximised by candidates that are **near-duplicates of the
analogs**, whereas the true molecule is usually a *derivative* (median Tanimoto to its best analog
is 0.83, most often ±CH₂ or ±O away). COCONUT works precisely *because* it does not contain those
near-duplicate decoys; any large isomer set supplies them in bulk and order statistics guarantee
several out-rank the truth.

### Does the fingerprint model rescue PubChem? No. (I checked.)

The obvious hypothesis is that PubChem only failed because analog-Tanimoto is the wrong scorer for
it, and that a model scoring each structure **intrinsically** would fix it. I tested exactly that
with the full four-channel ranker:

| candidate pool | Class-2 MRR |
|---|---|
| COCONUT + train only | **0.732** |
| \+ top-25 PubChem | 0.379 |
| \+ top-100 PubChem | 0.328 |
| \+ top-500 PubChem | 0.338 |

Still catastrophic. The arithmetic settles it: adding PubChem is net-positive only when

$$g \;>\; \frac{0.35\,\rho}{1-\rho}$$

where $\rho$ is your current pool recall and $g$ is the MRR you achieve on the molecules your pool
misses. At a plausible $\rho \approx 0.6$ that demands $g > 0.52$ — better than this ranker manages
on a *56*-candidate list, let alone 4,000. **Do not spend your time here.**

### What does work: targeted expansion

ChEBI + LIPID MAPS adds only **+8.8%** candidates but **+7–19%** coverage of the structures in the
public spectral libraries (ChEBI brings metabolites, LIPID MAPS brings lipids — both regions a
plant/microbe-focused NP database under-covers). Dilution cost is **−0.026** with the model channel
(it was −0.035 without), which the coverage gain should more than repay. Toggle with
`CFG.USE_BIO_DB`.

> **Open problem for you:** ~57% of truths sit within one or two common biosynthetic deltas
> (+CH₂, +O, +hexose, …) of their best analog. Expanding the pool with *targeted derivatives*
> should raise recall — but only once the ranker can tell **which** derivative. Note that
> positional isomers give *different* fragment-mass sets, so the in-silico fragmentation channel
> is the one with a chance of telling them apart.


### Building the other half of the pool, at runtime

The attached dataset contains **only the COCONUT half** of the candidate pool. The competition
rules forbid redistributing competition data to non-participants, and the training structures come
from `train.parquet` — so the notebook rebuilds that half itself, from the file you already have.

It costs ~5 minutes of RDKit on the Kaggle CPU and makes the pipeline fully reproducible: nothing
about the candidate pool is hidden inside a precomputed blob you cannot inspect.

The fingerprint is `ECFP4(4096) ‖ ECFP6(4096) ‖ RDKitFP(2048) ‖ MACCS(167)`, reduced to the
**6,930** bits whose frequency across training structures lies in [0.5%, 99.5%] (`fp_bits.npy`).
Rare bits carry no discrimination and common bits carry no information.


In [ ]:

# ===================================================================================
#  Candidate pool = COCONUT (attached, CC-BY) U training structures (rebuilt here).
#  Only the COCONUT half is redistributable, so the other half is computed at runtime.
# ===================================================================================
from rdkit import Chem, RDLogger
from rdkit.Chem import rdFingerprintGenerator, MACCSkeys
from rdkit.Chem.Descriptors import ExactMolWt
from multiprocessing import Pool as MPool
RDLogger.DisableLog('rdApp.*')

BITS = np.load(find('fp_bits.npy'))
_g = {}
def _fp_init():
    _g['m2'] = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=4096)
    _g['m3'] = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=4096)
    _g['rk'] = rdFingerprintGenerator.GetRDKitFPGenerator(fpSize=2048, maxPath=6)

def fp_and_mass(smi):
    if not _g: _fp_init()
    m = Chem.MolFromSmiles(smi)
    if m is None: return None
    try:
        fp = np.concatenate([_g['m2'].GetFingerprintAsNumPy(m).astype(np.uint8),
                             _g['m3'].GetFingerprintAsNumPy(m).astype(np.uint8),
                             _g['rk'].GetFingerprintAsNumPy(m).astype(np.uint8),
                             np.array(MACCSkeys.GenMACCSKeys(m), dtype=np.uint8)])[BITS]
        return fp, float(ExactMolWt(m))
    except Exception:
        return None

class Pool:
    """Candidates + fingerprints, sorted by exact mass."""
    def __init__(s, fp, mass, keys, smiles, nbits):
        o = np.argsort(mass)
        s._fp = fp[o]; s.mass = mass[o]
        s.keys = np.asarray(keys, dtype=object)[o]
        s.smiles = np.asarray(smiles, dtype=object)[o]
        s.nbits = nbits
        s.k2i = {k: i for i, k in enumerate(s.keys)}
    def window(s, t, ppm):
        a = np.searchsorted(s.mass, t*(1-ppm/1e6), 'left')
        b = np.searchsorted(s.mass, t*(1+ppm/1e6), 'right')
        return np.arange(a, b)
    def fps(s, idx):
        return np.unpackbits(np.asarray(s._fp[idx]), axis=1)[:, :s.nbits]

def build_pool():
    t0 = time.time()
    d = os.path.dirname(find('coco_fp.npy'))
    cm = pickle.load(open(d + '/coco_meta.pkl', 'rb'))
    co_fp = np.load(d + '/coco_fp.npy'); co_mass = np.load(d + '/coco_mass.npy')
    co_keys = np.asarray(cm['keys'], dtype=object); co_smis = np.asarray(cm['smiles'], dtype=object)
    print(f'COCONUT: {len(co_mass):,} structures', flush=True)

    # ChEBI + LIPID MAPS: small, high-precision, and covers mammalian/lipid metabolites that a
    # plant/microbe-focused NP database misses. +8.8% candidates for +7-19% coverage of the
    # structures in the public spectral libraries (see the pool section).
    if CFG.USE_BIO_DB:
        try:
            bd = os.path.dirname(find('bio_fp.npy'))
            bm = pickle.load(open(bd + '/bio_meta.pkl', 'rb'))
            bi_fp = np.load(bd + '/bio_fp.npy'); bi_mass = np.load(bd + '/bio_mass.npy')
            co_fp = np.vstack([co_fp, bi_fp]); co_mass = np.concatenate([co_mass, bi_mass])
            co_keys = np.concatenate([co_keys, np.asarray(bm['keys'], dtype=object)])
            co_smis = np.concatenate([co_smis, np.asarray(bm['smiles'], dtype=object)])
            print(f'+ ChEBI/LIPID MAPS: {len(bi_mass):,} structures', flush=True)
        except FileNotFoundError:
            print('ChEBI/LIPID MAPS dataset not attached - skipping', flush=True)

    tr = pq.read_table(TRAIN, columns=['inchikey14', 'normalized_smiles']).to_pandas()
    tr = tr.dropna().drop_duplicates('inchikey14')
    tr = tr[~tr.inchikey14.isin(set(co_keys))]
    print(f'training structures to fingerprint: {len(tr):,}  (~5 min)', flush=True)
    with MPool(4) as mp:
        res = mp.map(fp_and_mass, list(tr.normalized_smiles), chunksize=500)
    ok = [i for i, r in enumerate(res) if r is not None]
    tr_fp = np.packbits(np.stack([res[i][0] for i in ok]), axis=1)
    tr_mass = np.array([res[i][1] for i in ok])
    tr_keys = tr.inchikey14.values[ok]; tr_smi = tr.normalized_smiles.values[ok]

    fp = np.vstack([co_fp, tr_fp])
    mass = np.concatenate([co_mass, tr_mass])
    keys = np.concatenate([co_keys, tr_keys])
    smis = np.concatenate([co_smis, tr_smi])
    good = np.isfinite(mass)
    print(f'pool: {int(good.sum()):,} structures   ({time.time()-t0:.0f}s)', flush=True)
    return Pool(fp[good], mass[good], keys[good], smis[good], cm['nbits'])


### How the candidate pool was built (so you can rebuild or extend it)

The attached `casmi26-pool` dataset is derived entirely from the competition data plus one public
database, and is reproducible in ~10 minutes:

1. **COCONUT 2.0** (`coconut_csv-09-2026.zip`, CC-BY, <https://coconut.naturalproducts.net/download>)
   — parse `canonical_smiles`, keep the largest fragment (drops salts/counterions), reject charged
   species and anything outside 100–1300 Da or 5–120 heavy atoms → **462,028** unique InChIKey14.
2. **Training-set structures** from `train.parquet` → **275,810** unique InChIKey14.
3. Union, deduplicated on **InChIKey14** (the metric is stereochemistry-blind, so two spellings of
   one skeleton would only waste a slot) → **711,705**, sorted by exact mass.
4. Fingerprint per structure: `ECFP4(4096) ‖ ECFP6(4096) ‖ RDKitFP(2048) ‖ MACCS(167)`, then keep
   only bits whose frequency across the training structures lies in **[0.5%, 99.5%]** →
   **6,930 bits**, bit-packed. Shipping them precomputed is why this notebook needs **no RDKit**.

Swapping in your own database is a drop-in change: produce `pool_mass.npy`, `pool_fp.npy`
(bit-packed, same 6,930 bits) and `pool_meta.pkl` with `keys`/`smiles`, and everything downstream
works unchanged.

### Credits

* **Entropy similarity** — Li, Kind, Fiehn et al., *Nature Methods* 2021.
* **COCONUT 2.0** — Chandrasekhar, Steinbeck et al., *Nucleic Acids Research* 2025.
* **CSI:FingerID / fingerprint retrieval** — Dührkop, Böcker et al., *PNAS* 2015 (the `f·z`
  identity in the last section is the linear-algebra shortcut through their scoring function).
* Analog / mass-shifted matching is the standard **GNPS molecular-networking** idea, applied here
  as a *ranking prior over a candidate database* rather than for network visualisation.


In [ ]:
"""Single source of truth for candidate ranking features (used by local fitting AND the notebook).

Deliberately EXCLUDES any feature revealing pool provenance (src / np_likeness): in the Class-2
simulation the answer is always a training-library structure, so those columns leak.
"""
import numpy as np

N_ANALOG     = 200
P_SIM    = 3.0
N_FEAT   = 31

def _rank_norm(x):
    o=np.argsort(-x); r=np.empty(len(x)); r[o]=np.arange(len(x)); return r/max(1,len(x)-1)

def _z(x):
    s=x.std()
    return (x-x.mean())/s if s>1e-9 else np.zeros_like(x)

def rank_features(cand_fp, cand_lib, analog_fp, analog_sim, model_logits=None, frag=None):
    """cand_fp (nc,nbits), cand_lib (nc,) library similarity (0 if none),
       analog_fp (na,nbits), analog_sim (na,) descending,
       model_logits (nbits,) or None -> fingerprint-model evidence,
       frag (nc,) or None -> in-silico fragmentation explain-score (MetFrag-lite).
       Returns X (nc, N_FEAT)."""
    nc = cand_fp.shape[0]
    cf = cand_fp.astype(np.float32); cs = cf.sum(1)
    lv = np.asarray(cand_lib, np.float32)
    lvmax = float(lv.max()) if nc else 0.0
    if analog_fp is not None and len(analog_sim):
        af = analog_fp.astype(np.float32); asum = af.sum(1)
        inter = cf @ af.T
        tan = inter/(cs[:,None]+asum[None,:]-inter+1e-9)
        w = np.clip(np.asarray(analog_sim,np.float32),0,None)
        ap = (tan*(w**P_SIM)[None,:]).max(1)
        a1 = (tan*w[None,:]).max(1)
        best_tan = tan.max(1); top_tan = tan[:,0]; top_sim = float(w[0])
        mean_tan = (tan*(w**P_SIM)[None,:]).sum(1)/((w**P_SIM).sum()+1e-9)
    else:
        ap=a1=best_tan=top_tan=mean_tan=np.zeros(nc,np.float32); top_sim=0.0
    apmax = float(ap.max()) if nc else 0.0
    if model_logits is not None:
        raw = cf @ np.asarray(model_logits, np.float32)       # exact Bayes LL up to a constant
        nrm = raw/np.sqrt(np.maximum(cs,1.0))                 # length-corrected variant
        mfeat = [_z(raw), _rank_norm(raw), raw-raw.max(), _z(nrm), _rank_norm(nrm),
                 (raw==raw.max()).astype(np.float32)]
    else:
        mfeat = [np.zeros(nc,np.float32)]*6
    # cross-channel agreement: a genuine Class-1 hit should look good to the MODEL too.
    # When the library's best match also ranks high under f.z, the library evidence is
    # corroborated; when it does not, the library hit is probably a same-mass impostor.
    if model_logits is not None and nc:
        mr = _rank_norm(cf @ np.asarray(model_logits, np.float32))
        lbest = int(np.argmax(lv)) if lvmax > 0 else -1
        agree = float(1.0 - mr[lbest]) if lbest >= 0 else 0.0      # 1 = model also ranks it first
        abest = int(np.argmax(ap)) if apmax > 0 else -1
        agree_a = float(1.0 - mr[abest]) if abest >= 0 else 0.0
        xfeat = [lv*(1.0-mr), ap*(1.0-mr), np.full(nc, agree), np.full(nc, agree_a),
                 np.full(nc, agree*lvmax), np.full(nc, float(np.corrcoef(lv, -mr)[0,1]) if lv.std()>1e-9 else 0.0)]
    else:
        xfeat = [np.zeros(nc,np.float32)]*6
    if frag is not None:
        fr = np.asarray(frag, np.float32)
        ffeat = [fr, _rank_norm(fr), fr-fr.max() if nc else fr, _z(fr)]
    else:
        ffeat = [np.zeros(nc,np.float32)]*4
    return np.column_stack([
        lv, _rank_norm(lv), np.full(nc,lvmax), lv-lvmax, (lv>0).astype(float),
        ap, _rank_norm(ap), np.full(nc,apmax), ap-apmax,
        a1, best_tan, top_tan, mean_tan, np.full(nc,top_sim),
        np.full(nc, np.log(max(nc,1))),
        *mfeat, *ffeat, *xfeat,
    ]).astype(np.float32)


---

## 🧪 Third channel: in-silico fragmentation (MetFrag-lite)

Analog propagation borrows structure from a *neighbour*. This channel instead judges each candidate
**on its own merits**: break its bonds, see whether the resulting fragments can explain the peaks
that were actually observed.

For each candidate we break every single bond, and every *pair* of bonds, keep the connected
components, allow ±2 hydrogen rearrangements, and score the fraction of (square-rooted) peak
intensity that lands within `MZ_TOL` of some fragment ion.

| Channel | Class-2 MRR |
|---|---|
| in-silico fragmentation alone | 0.259 |
| analog propagation alone | 0.521 |
| **both (fixed blend)** | **0.545** |

The two are genuinely independent — their correlation on the *true* structures is only **0.058** —
which is exactly why combining them helps. It costs ~14 ms per candidate, so a few minutes for the
whole test set.

It is also the one channel that is **not** fooled by near-duplicates of an analog, which is why it
was worth testing whether it rescues a PubChem-sized candidate pool. It does not (see above) — but
it is a clean, training-free source of evidence, and a natural place to plug in a real fragmenter
(CFM-ID, SIRIUS) if you want to go further.


In [ ]:
"""MetFrag-lite: score a candidate by how much of the observed spectrum its bond-breaking
   fragments can explain. Independent evidence from analog propagation."""
import numpy as np
from rdkit import Chem, RDLogger
RDLogger.DisableLog('rdApp.*')

AMU = {'C':12.0,'H':1.00782503207,'N':14.0030740048,'O':15.9949146196,'P':30.97376163,
       'S':31.97207100,'F':18.99840322,'Cl':34.96885268,'Br':78.9183371,'I':126.904473,
       'Na':22.9897692809,'K':38.96370668,'Si':27.9769265325,'B':11.0093054,'Se':79.9165213}
H = AMU['H']; PROTON = H - 0.00054857990

def mol_graph(smi):
    m = Chem.MolFromSmiles(smi)
    if m is None: return None
    n = m.GetNumAtoms()
    w = np.zeros(n)
    for a in m.GetAtoms():
        w[a.GetIdx()] = AMU.get(a.GetSymbol(), 0.0) + a.GetTotalNumHs()*H
    if (w == 0).any(): return None
    bonds = [(b.GetBeginAtomIdx(), b.GetEndAtomIdx()) for b in m.GetBonds()]
    return w, bonds, n

def _components(n, bonds, drop):
    adj = [[] for _ in range(n)]
    for i,(a,b) in enumerate(bonds):
        if i in drop: continue
        adj[a].append(b); adj[b].append(a)
    seen = np.zeros(n, bool); comps=[]
    for s in range(n):
        if seen[s]: continue
        stack=[s]; seen[s]=True; cur=[s]
        while stack:
            u=stack.pop()
            for v in adj[u]:
                if not seen[v]: seen[v]=True; stack.append(v); cur.append(v)
        comps.append(cur)
    return comps

def fragment_masses(smi, max_breaks=2, max_bonds=34):
    """Neutral fragment masses from breaking 1 or 2 bonds."""
    g = mol_graph(smi)
    if g is None: return np.zeros(0)
    w, bonds, n = g
    nb = len(bonds)
    if nb == 0 or nb > max_bonds: return np.array([w.sum()])
    out = {w.sum()}
    for i in range(nb):
        for c in _components(n, bonds, {i}):
            out.add(float(w[c].sum()))
    if max_breaks >= 2:
        for i in range(nb):
            for j in range(i+1, nb):
                for c in _components(n, bonds, {i, j}):
                    out.add(float(w[c].sum()))
    return np.array(sorted(out))

def explain_score(frag_mass, peak_mz, peak_int, mode=1.0, tol=0.01, h_shifts=(-2,-1,0,1,2)):
    """Fraction of total (sqrt) intensity explained by some fragment ion."""
    if len(frag_mass) == 0 or len(peak_mz) == 0: return 0.0
    ion = []
    for dh in h_shifts:
        ion.append(frag_mass + dh*H + (PROTON if mode > 0 else -PROTON))
    ion = np.sort(np.concatenate(ion))
    w = np.sqrt(np.asarray(peak_int, float)); tot = w.sum()
    if tot <= 0: return 0.0
    idx = np.searchsorted(ion, peak_mz)
    ok = np.zeros(len(peak_mz), bool)
    for off in (-1, 0):
        k = np.clip(idx+off, 0, len(ion)-1)
        ok |= np.abs(ion[k]-peak_mz) <= tol
    return float(w[ok].sum()/tot)


In [ ]:

from multiprocessing import Pool as MPool

def _frag_masses(smi):
    try:  return fragment_masses(smi)
    except Exception: return np.zeros(0)

def frag_scores(cand_smiles, specs, mode, workers=4):
    """MetFrag-lite explain-score for every candidate, max over the molecule's spectra."""
    if not HAVE_RDKIT: return None
    with MPool(workers) as mp:
        frags = mp.map(_frag_masses, cand_smiles, chunksize=8)
    peaks = []
    for mz, it in specs:
        m2, i2 = _clean(np.asarray(mz, np.float32), np.asarray(it, np.float32),
                        CFG.INT_FLOOR, CFG.MAX_PEAKS, 1.0, False)
        peaks.append((np.asarray(m2, float), np.asarray(i2, float)))
    out = np.zeros(len(cand_smiles), np.float32)
    for j, f in enumerate(frags):
        out[j] = max((explain_score(f, a, b, mode=mode, tol=CFG.MZ_TOL) for a, b in peaks), default=0.0)
    return out


---

## 🧠 Channel 4: spectrum → fingerprint, and why ranking is *literally* a dot product

The first three channels all reason by **comparison** — to a library spectrum, to an analog, to a
candidate's own bond-breaking. This one predicts chemistry **directly from the spectrum**: a
transformer reads the peaks and outputs a 6,930-bit molecular fingerprint.

To rank a candidate with fingerprint $f$ under predicted per-bit logits $z$, the natural score is
the Bayes log-likelihood of its bits:

$$\sum_i \big[f_i\log\sigma(z_i) + (1-f_i)\log\sigma(-z_i)\big]$$

Now use the identity $\log\sigma(z)-\log\sigma(-z) = z$ (exactly, for all $z$). Splitting the sum:

$$= \sum_i f_i\big[\log\sigma(z_i)-\log\sigma(-z_i)\big] + \sum_i\log\sigma(-z_i)
  \;=\; \boxed{f\cdot z} \;+\; \underbrace{\textstyle\sum_i\log\sigma(-z_i)}_{\text{identical for every candidate}}$$

**Ranking by the full Bayesian score is a plain dot product with the raw logits.** No sigmoid, no
calibration, no temperature. Two consequences:

1. Inference is one matrix multiply.
2. The ranking objective can be trained **end-to-end**: each training spectrum is scored against
   63 decoy structures sampled from the *same ±10 ppm mass window* — precisely the competitors it
   will face at inference — under a softmax cross-entropy on $f\cdot z$.

### What it is worth

| Channel (alone) | Class-2 MRR@25 |
|---|---|
| in-silico fragmentation | 0.259 |
| **fingerprint model** | **0.468** |
| analog propagation | 0.521 |
| analog + model | **0.567** |
| all four, via the calibrated ranker | **0.612** |

The model is nearly as strong as analog propagation *by itself*, and because it reasons from
different evidence the two combine rather than overlap.

### Training notes (the mistake worth avoiding)

With only ~276k distinct structures, this model **memorises fast**. My first run looked healthy and
then quietly rotted:

| step | 9k | 12k | 40k | 69k |
|---|---|---|---|---|
| held-out hard-negative top-1 | 0.446 | **0.457** | ~0.30 | **0.127** |

Training loss kept improving the whole time. If you save the *last* checkpoint — as I did — you
ship a model three times worse than the one you had at step 12k. Two fixes are in the training
script: **keep the best-by-validation checkpoint**, and **augment** (peak dropout, intensity jitter,
±5 ppm m/z noise), which pushed the peak to 0.467 and moved it out to ~20k steps.


In [ ]:
"""Spectrum -> molecular fingerprint model (CSI:FingerID-style neural ranker)."""
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F, math

MAX_PEAKS = 128
ADDUCT_LIST = ["[M+H]+","[M+NH4]+","[M+Na]+","[M+K]+","[M-H2O+H]+","[M-2H2O+H]+","[M]+",
               "[M-H]-","[M-H2O-H]-","[M+CH2O2-H]-","[M+C2H4O2-H]-","[M+Cl]-","[M]-",
               "[M+2H]2+","[M-2H]-","[2M+H]+","[2M+Na]+","[2M+NH4]+","[2M-H]-","[2M+K]+",
               "[2M+CH2O2-H]-","[2M+C2H4O2-H]-","[2M+Na-2H]-","[M+Na-2H]-","[M-H2O]+","<unk>"]
ADDUCT_IX = {a:i for i,a in enumerate(ADDUCT_LIST)}
INSTR_LIST = ["timsTOF","Orbitrap","QTOF","IT","other"]
INSTR_IX = {a:i for i,a in enumerate(INSTR_LIST)}

def instr_family(s):
    if s is None: return 4
    t = str(s).lower()
    if 'timstof' in t: return 0
    if 'orbitrap' in t or 'qft' in t or 'ftms' in t or 'hybrid ft' in t or 'itft' in t or 'exactive' in t: return 1
    if 'tof' in t: return 2
    if 'trap' in t or 'qq' in t: return 3
    return 4

def prep_peaks(mz, inten, prec_mz, max_peaks=MAX_PEAKS, floor=1e-3, win=50.0, per_win=8):
    """Filter -> window-diversified top-N -> sort by m/z. Returns (mz, sqrt-intensity)."""
    mz = np.asarray(mz, np.float64); it = np.asarray(inten, np.float64)
    if len(mz)==0: return np.zeros(0,np.float32), np.zeros(0,np.float32)
    keep = (mz <= prec_mz + 1.5)
    mz, it = mz[keep], it[keep]
    if len(mz)==0: return np.zeros(0,np.float32), np.zeros(0,np.float32)
    mx = it.max()
    if mx <= 0: return np.zeros(0,np.float32), np.zeros(0,np.float32)
    keep = it >= floor*mx
    mz, it = mz[keep], it[keep]
    if len(mz) > max_peaks:
        # keep the top `per_win` peaks inside each `win` Da bucket, then global top-N
        order = np.argsort(-it)
        bucket = (mz//win).astype(np.int64)
        cnt = {}; sel=[]
        for i in order:
            b = bucket[i]; c = cnt.get(b,0)
            if c < per_win: cnt[b]=c+1; sel.append(i)
        sel = np.array(sel)
        if len(sel) > max_peaks:
            sel = sel[np.argsort(-it[sel])[:max_peaks]]
        elif len(sel) < max_peaks:
            rest = np.array([i for i in order if i not in set(sel.tolist())])
            need = max_peaks-len(sel)
            if len(rest): sel = np.concatenate([sel, rest[:need]])
        mz, it = mz[sel], it[sel]
    o = np.argsort(mz)
    mz, it = mz[o], it[o]
    v = np.sqrt(it/it.max())
    return mz.astype(np.float32), v.astype(np.float32)

class SinEmb(nn.Module):
    """Log-spaced sinusoidal embedding for m/z values (Voronov et al.)."""
    def __init__(self, dim, lo=-2.0, hi=3.2, power=1.0):
        super().__init__()
        n = dim//2
        wav = torch.pow(10.0, (hi-lo)*torch.pow(torch.linspace(0,1,n), power) + lo)
        self.register_buffer('inv', (2*math.pi)/wav)
    def forward(self, x):                      # x: (...,)
        a = x.unsqueeze(-1) * self.inv
        return torch.cat([torch.sin(a), torch.cos(a)], -1)

class Block(nn.Module):
    def __init__(self, d, h, drop):
        super().__init__(); self.h=h
        self.n1=nn.LayerNorm(d); self.qkv=nn.Linear(d,3*d); self.o=nn.Linear(d,d)
        self.n2=nn.LayerNorm(d)
        self.ff=nn.Sequential(nn.Linear(d,4*d), nn.GELU(), nn.Dropout(drop), nn.Linear(4*d,d))
        self.drop=nn.Dropout(drop)
    def forward(self, x, pad):
        B,N,D=x.shape; y=self.n1(x)
        q,k,v = self.qkv(y).view(B,N,3,self.h,D//self.h).permute(2,0,3,1,4)
        m = (~pad)[:,None,None,:]                       # True = attend
        a = F.scaled_dot_product_attention(q,k,v, attn_mask=m)
        x = x + self.drop(self.o(a.transpose(1,2).reshape(B,N,D)))
        return x + self.drop(self.ff(self.n2(x)))

class FPNet(nn.Module):
    def __init__(self, nbits, d=512, layers=6, heads=8, drop=0.1):
        super().__init__()
        self.d=d
        self.mz_emb  = SinEmb(d)
        self.nl_emb  = SinEmb(d)
        self.pk = nn.Linear(2*d+1, d)
        self.prec_emb = SinEmb(d)
        self.ad = nn.Embedding(len(ADDUCT_LIST), d)
        self.ins = nn.Embedding(len(INSTR_LIST), d)
        self.gl = nn.Linear(d+3, d)
        self.blocks = nn.ModuleList([Block(d,heads,drop) for _ in range(layers)])
        self.norm = nn.LayerNorm(d)
        self.head = nn.Sequential(nn.Linear(2*d, 2048), nn.GELU(), nn.Dropout(drop), nn.Linear(2048, nbits))
    def forward(self, mz, it, pad, prec, ad, ins, ce, mode):
        B,N = mz.shape
        nl = (prec[:,None] - mz).clamp(min=0)
        p = self.pk(torch.cat([self.mz_emb(mz), self.nl_emb(nl), it.unsqueeze(-1)], -1))
        g = self.gl(torch.cat([self.prec_emb(prec),
                               (ce/100.0).unsqueeze(-1), mode.unsqueeze(-1),
                               torch.log1p(prec).unsqueeze(-1)/10.0], -1)) + self.ad(ad) + self.ins(ins)
        x = torch.cat([g.unsqueeze(1), p], 1)
        pad = torch.cat([torch.zeros(B,1,dtype=torch.bool,device=pad.device), pad], 1)
        for b in self.blocks: x = b(x, pad)
        x = self.norm(x)
        cls = x[:,0]
        msk = (~pad[:,1:]).float().unsqueeze(-1)
        mean = (x[:,1:]*msk).sum(1)/msk.sum(1).clamp(min=1)
        return self.head(torch.cat([cls, mean], -1))


In [ ]:

# ===================================================================================
#  Channel 4: spectrum -> molecular fingerprint (CSI:FingerID-style), ranked by f . z
# ===================================================================================
import torch
_MODEL = None
def load_model():
    """Optional. If the weights dataset is not attached, everything still runs without it."""
    global _MODEL
    import glob
    paths = sorted(glob.glob('/kaggle/input/**/fp_*.pt', recursive=True))
    if not paths:
        print('fingerprint model not attached - running with 3 channels'); return None
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    # Each model is fed the input distribution it was TRAINED on. 'fp_merged_*' saw fused
    # multi-spectrum inputs (--merge_p 0.6) and wants one merged peak list; 'fp_single_*' saw
    # one spectrum at a time. Feeding either the wrong way costs ~0.02 Class-2 MRR:
    #   m1: merged 0.4896 / per-spectrum 0.4687     s2: per-spectrum 0.4799 / merged 0.4597
    single, merged = [], []
    for pth in paths:
        ck = torch.load(pth, map_location='cpu', weights_only=False)
        net = FPNet(ck['nbits'], d=ck['d'], layers=ck['layers']).to(dev).eval()
        net.load_state_dict(ck['model'])
        (merged if 'merged' in pth.split('/')[-1] else single).append(net)
        print(f'  loaded {pth.split("/")[-1]}: d={ck["d"]} layers={ck["layers"]} step={ck.get("step")}')
    print(f'fingerprint models: {len(single)} single-input, {len(merged)} merged-input, on {dev}')
    _MODEL = (single, merged, dev, ck['nbits'])
    return _MODEL

def _merge_peaks(sub):
    """All of a molecule's peaks collapsed into one pseudo-spectrum (near-duplicate m/z merged,
       keeping the stronger peak). A second view of the same molecule."""
    mz = np.concatenate([np.asarray(r.ms2_mzs, float) for r in sub.itertuples()])
    it = np.concatenate([np.asarray(r.ms2_normalized_intensities, float) /
                         max(float(np.asarray(r.ms2_normalized_intensities, float).max()), 1e-9)
                         for r in sub.itertuples()])
    o = np.argsort(mz); mz, it = mz[o], it[o]
    keep = np.ones(len(mz), bool)
    for j in range(1, len(mz)):
        if mz[j]-mz[j-1] < 0.005:
            if it[j] >= it[j-1]: keep[j-1] = False
            else: keep[j] = False
    return mz[keep], it[keep]

@torch.no_grad()
def model_logits(sub):
    """Two fusion views, averaged: (a) per-spectrum logits averaged, (b) one merged peak list.
       Measured on the Class-2 holdout: (a) 0.468, (b) 0.459, mean of both 0.475."""
    if _MODEL is None: return None
    single, merged, dev, nbits = _MODEL
    out = []
    if single:
        za = _logits_from(sub, single)
        if za is not None: out.append(za)
    if merged:
        mz, it = _merge_peaks(sub)
        r0 = next(sub.itertuples())
        zb = _logits_raw([(mz, it)], merged, float(np.median(sub.precursor_mz)), r0.adduct,
                         r0.instrument_type, 25.0,
                         float(np.mean([1.0 if m=='positive' else -1.0 for m in sub.ionization_mode])))
        if zb is not None: out.append(zb)
    return np.mean(out, axis=0) if out else None

@torch.no_grad()
def _logits_from(sub, nets):
    if _MODEL is None: return None
    dev = _MODEL[2]
    rows = list(sub.itertuples())
    P = [prep_peaks(r.ms2_mzs, r.ms2_normalized_intensities, float(r.precursor_mz)) for r in rows]
    P = [(a,b) for a,b in P if len(a)]
    if not P: return None
    B = len(P); N = max(len(a) for a,_ in P)
    mz=np.zeros((B,N),np.float32); it=np.zeros((B,N),np.float32); pad=np.ones((B,N),bool)
    for i,(a,b) in enumerate(P):
        mz[i,:len(a)]=a; it[i,:len(b)]=b; pad[i,:len(a)]=False
    def ce_of(r):
        v=r.collision_energy_ev
        try: return float(np.mean(np.atleast_1d(v))) if v is not None and len(np.atleast_1d(v)) else 25.0
        except Exception: return 25.0
    T=lambda x: torch.as_tensor(x, device=dev)
    args = (T(mz), T(it), T(pad),
            T(np.array([float(r.precursor_mz) for r in rows[:B]],np.float32)),
            T(np.array([ADDUCT_IX.get(r.adduct, ADDUCT_IX['<unk>']) for r in rows[:B]])),
            T(np.array([instr_family(r.instrument_type) for r in rows[:B]])),
            T(np.array([ce_of(r) for r in rows[:B]],np.float32)),
            T(np.array([1.0 if r.ionization_mode=='positive' else -1.0 for r in rows[:B]],np.float32)))
    # average over the ensemble, then over the molecule's spectra
    return np.mean([n(*args).float().mean(0).cpu().numpy() for n in nets], axis=0)

@torch.no_grad()
def _logits_raw(pairs, nets, prec, adduct, instrument, ce, mode):
    """Same forward pass for an explicitly supplied peak list."""
    if _MODEL is None: return None
    dev = _MODEL[2]
    P=[prep_peaks(mz, it, prec) for mz, it in pairs]
    P=[(a,b) for a,b in P if len(a)]
    if not P: return None
    B=len(P); N=max(len(a) for a,_ in P)
    mz=np.zeros((B,N),np.float32); it=np.zeros((B,N),np.float32); pad=np.ones((B,N),bool)
    for i,(a,b) in enumerate(P):
        mz[i,:len(a)]=a; it[i,:len(b)]=b; pad[i,:len(a)]=False
    T=lambda x: torch.as_tensor(x, device=dev)
    args=(T(mz),T(it),T(pad), T(np.full(B,prec,np.float32)),
          T(np.full(B, ADDUCT_IX.get(adduct, ADDUCT_IX['<unk>']))),
          T(np.full(B, instr_family(instrument))),
          T(np.full(B, ce, np.float32)), T(np.full(B, mode, np.float32)))
    return np.mean([n(*args).float().mean(0).cpu().numpy() for n in nets], axis=0)


In [ ]:

# ===================================================================================
#  Optional pool expansion: PubChem isomers inside the same ppm window.
#  This is the one lever the local holdout CANNOT score. Every validation answer is
#  already in the pool (recall 100% at +-10 ppm), so adding a database can only ever
#  show up as dilution -- never as the recall it buys. The decision is therefore made
#  by algebra, not by validation:
#      adding pays  iff  rho' * d > rho
#  where rho is current pool recall, rho' the expanded recall, and d the dilution
#  factor. Measured COCONUT coverage of natural products brackets rho at 0.38-0.99,
#  and the break-even sits at 0.52 -- inside the bracket. So it gets a submission.
#  Dilution is controlled by admitting only the top PC_TOPK isomers by the model's
#  f.z, which is real evidence, rather than all of them (that cost d ~ 0.52).
# ===================================================================================
import glob, os

PC = None
def load_pcstore():
    global PC
    hits = sorted(glob.glob('/kaggle/input/**/mass_sorted.npy', recursive=True))
    if not hits or not CFG.PC_TOPK:
        print('PubChem store not attached - pool expansion off'); return None
    d = os.path.dirname(hits[0])
    PC = dict(ms=np.load(d + '/mass_sorted.npy', mmap_mode='r'),
              order=np.load(d + '/order.npy', mmap_mode='r'),
              off=np.load(d + '/off.npy', mmap_mode='r'),
              ln=np.load(d + '/len.npy', mmap_mode='r'),
              fh=open(d + '/smiles.txt', 'rb'))
    print(f'PubChem store: {len(PC["ms"]):,} NP-formula structures, mass-indexed')
    return PC

def pc_window(target, ppm, cap):
    lo = np.searchsorted(PC['ms'], target*(1-ppm/1e6), 'left')
    hi = np.searchsorted(PC['ms'], target*(1+ppm/1e6), 'right')
    idx = np.asarray(PC['order'][lo:hi])
    if len(idx) > cap:                      # even sample, not a mass-biased prefix
        idx = idx[np.linspace(0, len(idx)-1, cap).astype(np.int64)]
    return idx

def pc_smiles(idx):
    out = []; o = np.asarray(PC['off'][idx]); l = np.asarray(PC['ln'][idx])
    for a, b in zip(o, l):
        PC['fh'].seek(int(a)); out.append(PC['fh'].read(int(b)).decode('ascii', 'ignore'))
    return out

def _fp_canon(smi):
    """Fingerprint + a canonical, stereo-free SMILES so we can drop entries the pool already has."""
    if not _g: _fp_init()
    m = Chem.MolFromSmiles(smi)
    if m is None: return None
    try:
        Chem.RemoveStereochemistry(m)
        return Chem.MolToSmiles(m), np.concatenate([
            _g['m2'].GetFingerprintAsNumPy(m).astype(np.uint8),
            _g['m3'].GetFingerprintAsNumPy(m).astype(np.uint8),
            _g['rk'].GetFingerprintAsNumPy(m).astype(np.uint8),
            np.array(MACCSkeys.GenMACCSKeys(m), dtype=np.uint8)])[BITS]
    except Exception:
        return None

def pubchem_extra(target, zlog, known, workers=4):
    """Top-PC_TOPK PubChem isomers by f.z that the pool does not already contain."""
    if PC is None or zlog is None or not CFG.PC_TOPK: return None, []
    idx = pc_window(target, CFG.PPM_WIN, CFG.PC_WINCAP)
    if len(idx) == 0: return None, []
    smis = pc_smiles(idx)
    with MPool(workers) as mp:
        res = mp.map(_fp_canon, smis, chunksize=64)
    fps = []; sms = []
    for r in res:
        if r is None: continue
        cs, f = r
        if cs in known: continue
        known.add(cs); fps.append(f); sms.append(cs)
    if not fps: return None, []
    F = np.stack(fps).astype(np.uint8)
    z = F.astype(np.float32) @ np.asarray(zlog, np.float32)
    k = np.argsort(-z)[:CFG.PC_TOPK]
    return F[k], [sms[i] for i in k]


---

## ⚠️ The candidate cap: a recall bug that hides from validation

Several popular forks of this problem shrink the candidate list before ranking, keeping the best
`80` by

```python
coarse = library_sim * 100.0 - abs(pool_mass[cand] - target)
```

It reads like a sharper version of "tighter windows win" — prefer the library hits, then the
closest in mass. It is worth stepping through why it is not.

**A Class-2 molecule has `library_sim = 0` by definition.** That is what Class 2 *means*: the
structure exists in a database but no public reference spectrum does. So the `library_sim * 100`
term is identically zero for exactly the molecules the whole notebook is trying to identify, and
their ordering collapses to `-|Δmass|` — mass proximity *inside a window that was already selected
by mass*. At ±8.5 ppm around 500 Da the entire window spans 0.0085 Da; ranking candidates by where
they fall inside it is close to ranking them at random.

Truth retention on the Class-2 holdout, measured directly (does the correct structure survive the
cap at all?):

| cap | Class-1 retention | Class-2 retention |
|---|---|---|
| none | 0.992 | **0.992** |
| 400, by mass | 0.992 | **0.992** |
| 200, by mass | 0.992 | 0.952 |
| 80, by model `f·z` | 0.992 | 0.900 |
| **80, by mass** | 0.992 | **0.752** |

A cap of 80 discards **a quarter of the answers that were reachable**, before the ranker ever sees
them. No amount of ranking quality recovers them — MRR is bounded above by retention.

**Why this survives validation.** Look at the Class-1 column: it is 0.992 at *every* setting,
because `library_sim * 100` dominates and library hits are always kept. If you sanity-check a cap
on the channel that is easy to check, it looks free. The damage is confined to the class that has
no library evidence to protect it — and that class is ~⅔ of the scoring weight
(`B ≈ 0.271` vs `A ≈ 0.162`).

Running the full four-channel ranker either way, on the same seeds and the same features:

| | Class-1 MRR | Class-2 MRR | predicted LB |
|---|---|---|---|
| no cap | 0.922 | **0.789** | **0.363** |
| cap 80 by model `f·z` | 0.889 | 0.696 | 0.333 |
| cap 80 by mass | 0.896 | 0.603 | 0.308 |

**The arithmetic closes.** If retention were the *only* damage, capped MRR should be
uncapped MRR × retention ratio:

* cap 80 by mass: `0.789 × (0.752/0.992) = 0.598` — measured **0.603**
* cap 80 by `f·z`: `0.789 × (0.900/0.992) = 0.716` — measured **0.696**

So essentially the entire loss is answers that never reached the ranker, plus a small residual.
That residual has a cause too: `rank_features` contains **set-relative** features (`_rank_norm`
divides by `len(cand)-1`, `_z` subtracts the set mean). The ranker was fitted on *uncapped*
candidate sets, so capping at inference silently rescales those features. Removing the cap fixes
a train/inference mismatch as well as the recall loss.

The cap here is kept only as a **runtime guard** on in-silico fragmentation (~14 ms/candidate), set
to `500` — the median window holds 52 candidates and the largest observed holds 401, so it
never fired once across the 248 validation queries (at 80 it fired on 97 of them, 39%). When it
does fire it ranks by the model's `f·z`, which is actual evidence, rather than by mass, which is
not.

**This was not a hypothetical.** Adopting that fork's constants as a bundle took this notebook
from **0.335 → 0.282** on the public leaderboard. Decomposed against the calibrated objective:

| change | predicted ΔLB | source |
|---|---|---|
| candidate cap 80 by mass | **−0.055** | retention 0.992 → 0.752 |
| window 10 ppm → 8.5 ppm | −0.010 | retention 1.000 → 0.992 |
| 36k-step → 50k-step merged model | +0.008 | Class-2 channel 0.490 → 0.517 |
| **net** | **−0.057** | **observed −0.053** |

Four of the five constants were harmless (`N_ANALOG` 80→100 moves predicted LB by 0.0004; `SIM_POWER`
3→4 is a genuine small win). Only one was a bug — and it travelled attached to the reputation of the
others. **Adopt constants one at a time, and make each one show you its own measurement.**

The general lesson is worth more than the fix: **any pruning heuristic keyed on a feature that is
structurally absent for one class will silently delete that class.** Check retention per class, not
aggregate MRR.


---

## Combining the evidence: calibrate, don't hand-weight

Library similarity and analog evidence are on incomparable scales. Adding them with fixed weights
fails badly — putting library similarity in with weight 1.0 drops Class-2 MRR from **0.52 → 0.27**,
because for a Class-2 molecule *every* library hit is a wrong same-mass isomer.

Instead a small gradient-boosted model maps the evidence to a calibrated
**P(candidate is the answer)**, trained on a Class-1 simulation and a Class-2 simulation mixed at
the measured base rate.

### ⚠️ Two leaks to avoid if you build your own holdout

Masking a structure's spectra out of the library leaks in two ways, and both produced large fake
gains before being caught:

1. **Pool provenance.** Masking removes *evidence*, not *membership* — the answer is still a
   training-library structure in the pool. Feeding the ranker `src` (train vs COCONUT-only) or
   `np_likeness` (NaN for exactly the training structures) scored **0.94** on a task that honestly
   scores 0.55.
2. **Query in library.** A Class-1 simulation must drop the query's own *source library*, not just
   its InChIKey — otherwise it retrieves the identical spectrum and reports a perfect 1.000.

### Choosing the class weight

`CFG.W1` is **not** the Class-1 share. A Class-2 molecule is only worth something if the pool
actually contains it, so its value is discounted by pool recall. Solving the two leaderboard
readings —

* library-only `0.151` with Class-1 MRR `0.93` ⟹ Class-1 value `0.162`
* full ranker `0.233` with Class-1 MRR `0.73` ⟹ Class-2 value `0.22`

— gives `W1 = 0.162 / (0.162 + 0.22) ≈ 0.42`, roughly **2.6× more weight on Class 1** than the raw
share suggests. Sweeping it against that objective:

| `W1` | Class-1 MRR | Class-2 MRR | predicted LB |
|---|---|---|---|
| 0.16 *(raw share)* | 0.735 | 0.545 | 0.239 |
| **0.42** | **0.873** | **0.511** | **0.254** |
| 0.85 | 0.910 | 0.460 | 0.249 |

If your pool recall differs from mine, re-solve for `W1` — it is the single most leveraged constant
in the notebook.


---

## 🪤 When validation stops predicting the leaderboard — and how far you can actually diagnose it

This one cost a leaderboard place, and the honest version of the story is more useful than the
tidy one I first wrote.

**The observation.** Two model channels, measured on both local holdouts and on the leaderboard:

| model channel | 250-NP holdout | 569-obscure holdout | **public LB (whole config)** |
|---|---|---|---|
| `m1@24k` merged + `s2` per-spectrum, averaged | 0.4904 | 0.5579 | **0.335** |
| `m1@36k` merged alone | **0.5173** | **0.5624** | **0.311** |

Both local sets prefer the later checkpoint. The leaderboard preferred the configuration built on
the earlier pair, by 0.024. So the local numbers did not transfer.

**Those two scores came from configurations differing in four ways**, not one: the model channel,
`N_ANALOG` 80→100, `SIM_POWER` 3→4, and a newly added candidate cap. That is the candidate-cap
mistake again — several changes at once, then reasoning as if there had been one. So rather than
guess, I spent a submission on a proper **single-variable A/B**: two runs identical in every
respect except which model dataset was attached.

| configuration | public LB |
|---|---|
| `m1@24k` merged + `s2` per-spectrum | **0.330** |
| `m1@36k` merged alone | **0.311** |

**+0.019 for the two-model channel**, isolated. That also clears the analog constants: they were
innocent, and the model channel carried the entire regression. It is the largest single effect I
have measured on this leaderboard — and both local holdouts rank it the wrong way round.

**A mechanism I proposed, tested, and had to drop.** The obvious explanation is checkpoint
selection: the trainer saves on `if va > best_val`, so later checkpoints are the ones that won
more comparisons against a held-out split built from the same natural-product libraries my
holdouts come from. Selection pressure pointing at my own measuring stick.

It is a good story and the arithmetic kills it. The early-stopping check draws a **fresh random
2048-spectrum sample from 109,985 validation spectra** each time, so any given spectrum appears in
about **1.9%** of checks — roughly once across the whole run — and the decision it feeds is a
single aggregate scalar, about fifty times total. That is nowhere near enough selection pressure
to move a specific 250-structure subset by 0.027. I checked the ordinary leak too: all 250
validation structures sit in the model's held-out half, none in its training half.

So: real effect, mechanism unproven.

**What survives, and it is the part worth keeping.** Both holdouts are built the same way — mask a
structure's spectra out of a library that is overwhelmingly natural-product and timsTOF-heavy — and
the hidden test's Class-2 molecules are by definition ones with *no* reference spectra anywhere.
Those are different distributions, and a local MRR difference of 0.02–0.03 between two model
channels simply does not survive the trip. Note that the two local sets do not even agree with each
other on how big the gap is (0.027 vs 0.005), which is itself a warning that neither is measuring
what the leaderboard measures.

**What this notebook does about it**

1. **Ships the configuration that actually scored**, not the one that validates best. Every
   constant here is at the value from the highest-scoring submission, and where local validation
   disagrees, the local number loses.
2. **Keeps the two channels distinguishable.** Local validation is still used for things it *can*
   measure — candidate **retention** is a set-membership fact, not a model opinion, and the
   candidate-cap section above is built entirely on it. Prefer measurements that do not depend on
   a model's taste.
3. **Changes one thing per submission.** Both regressions in the results table came from bundling.

If you fork this and can afford the GPU time, the clean fix is a **three-way split**: train, a
model-selection split, and an evaluation split from a *different* library or instrument family
than either. Then a local number means something again.


---

## ⚠️ Before you trust any leaderboard delta: measure your noise floor

I submitted the **same notebook version twice** and got **0.292** and **0.298**.

Same code, same data, same everything. The culprit is one unset argument:
`HistGradientBoostingClassifier` defaults to `random_state=None`, so every fit bins and splits
differently. Reproducing it locally over five seeds:

| seed | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| predicted LB | 0.3033 | 0.3002 | 0.2961 | 0.3003 | 0.2977 |

**Range 0.0072** — which matches the 0.006 I saw on the actual leaderboard.

This is worth pausing on, because in a competition where the whole public leaderboard spans about
0.15–0.34, a **0.005 difference is indistinguishable from your ranker's seed**. I had to retract one
of my own conclusions because of it: I reported that adding ChEBI + LIPID MAPS *hurt*, based on
0.299 → 0.295. That gap sits inside the noise band. It is not evidence, and I should not have stated
it as firmly as I did.

**Why the two-model channel wins, and what that predicts.** The two models are not redundant —
they are fed *different input representations*. `s2` was trained one spectrum at a time and is fed
one spectrum at a time; `m1` was trained on fused multi-spectrum inputs (`--merge_p 0.6`) and is fed
a single merged peak list. Feeding either the other's view costs about 0.02, which is why
`load_model` routes by filename. So the ensemble is not "two draws of the same estimator" — it is
two genuinely different readings of the same spectra.

That distinction makes a testable prediction: adding models *within* a view should give ordinary,
much smaller ensembling gains, while having **both** views is the large, structural win. Worth
knowing before you spend GPU hours training a third model of the same kind.

**What this notebook does about it**

1. **Pin the seed** (`CFG.SEEDS`) so a rerun is reproducible.
2. **Average over seeds** — variance falls as ~1/√n *and* the mean improves slightly, because
   averaging decorrelated trees is just bagging.
3. **Average over class priors too** (`CFG.W1_PRIORS`) — and here is the mistake I made, left in
   because it is the most repeatable mistake in this whole competition.

   After the model channel improved I re-swept `W1` and tree depth, and got a beautiful result: a
   clean unimodal curve peaking at `W1 = 0.45`, a narrow plateau `(.40,.45,.50)` beating the shipped
   `(.30,.60)` by 0.008, and depth 6 → 8 → 10 adding another 0.007, each step improving **both**
   classes rather than trading one for the other. I checked it against the seed-noise band. I shipped
   it. It scored **0.330 against the 0.335 it replaced.**

   The sweep trained on all 819 query groups and then evaluated on 250 of them. **Those 250 were
   training rows.** Deeper trees and finer priors fit the evaluation queries better because the
   ranker had already seen them — I was measuring capacity to memorise my own test. The "both
   classes improve together" signal I trusted as evidence against noise is exactly what in-sample
   overfitting produces, since both classes share the same query groups.

   `make_ranker3.py` had always done this correctly, with 5 folds **held out by query group**
   (`folds[G] != f`). My quick sweep dropped the fold logic to go faster, and I did not notice that
   this changed what the number meant. The shipped values are the cross-validated ones.

   **The two rankings, side by side.** Same configs, same features, same seeds — the only
   difference is whether the evaluation queries were also training rows:

   | ranker config | in-sample sweep | **5-fold, held out by query** | actual LB |
   |---|---|---|---|
   | `(.30,.60)` depth 6 | 0.3477 | 0.3024 | **0.335** |
   | `(.30,.60)` depth 10 | 0.3479 | **0.3042** | — |
   | `(.45)` depth 6 | — | 0.3023 | — |
   | `(.40,.45,.50)` depth 6 | 0.3530 | 0.3003 | — |
   | `(.40,.45,.50)` depth 10 | **0.3603** | 0.3017 | **0.330** |

   The in-sample column ranks the configurations **almost exactly backwards**. Its top choice is
   the honest column's second-worst, and the two rows I actually submitted confirm the honest
   column's ordering, not the sweep's. In-sample numbers are also inflated by ~15% in absolute
   terms, but the inflation is not the problem — the *re-ordering* is.

   There is a real finding buried in there, and it only became visible once the evaluation was
   honest: **depth 10 is genuinely better, but only with the original priors** (0.3024 → 0.3042).
   The narrow plateau was the harmful half of what I bundled, and it was harmful enough to cancel
   the depth gain and then some. Two changes, opposite signs, shipped together — which is the same
   mistake as the candidate cap, made again, by me, after I had written the section warning about it.

   **If you take one thing from this notebook, take this:**   **If you take one thing from this notebook, take this:** when you tune a ranker, hold out by
   *query*, not by row. Every candidate of one molecule must be on the same side of the split.
   Otherwise the model sees the answer's neighbours at training time and the number you get back is
   not a prediction of anything.

**The local noise band.** Fitting the same config with four *disjoint* seed sets (0-3, 4-7, 8-11,
12-15) at W1 = 0.45 gives 0.3811 / 0.3752 / 0.3784 / 0.3762 — a spread of **0.0059** around a mean
of 0.3777. That is the bar: an improvement smaller than ~0.006 on a single 4-seed run is not
distinguishable from the seed. The W1 curve above clears it because it is a *smooth shape across
seven points*, not one lucky measurement — which is the cheapest way to tell signal from noise
without spending submissions.

**What you should do about it:** before you accept that some change bought you +0.004, submit the
*unchanged* notebook twice and see how far apart the two scores land. It costs two submissions and
it will save you from optimising noise for a week.


In [ ]:

# ===================================================================================
#  Calibrated ranker.  Fitted here, in-notebook, from shipped simulation features so
#  the whole thing is reproducible and you can retune W1 in one line.
# ===================================================================================
from sklearn.ensemble import HistGradientBoostingClassifier
z = np.load(find('rank_train.npz'))
NFEAT = z['X'].shape[1]

# ---------------------------------------------------------------------------------
#  ⚠️  HistGradientBoostingClassifier defaults to random_state=None.
#  I submitted the SAME notebook version twice and got 0.292 and 0.298 -- a 0.006
#  spread from the ranker's seed alone (5 local seeds span 0.2961-0.3033). If you are
#  chasing a 0.005 leaderboard difference in this competition, you may be chasing noise.
#  Fix: pin the seed AND average over several, which removes the variance and lifts the mean.
# ---------------------------------------------------------------------------------
RANKERS = []
for w1 in CFG.W1_PRIORS:                  # a NARROW plateau around the swept peak -- see CFG
    W = np.where(z['M'] == 0, w1, 1.0 - w1)
    for sd in CFG.SEEDS:
        m = HistGradientBoostingClassifier(random_state=sd, **CFG.GBM)
        m.fit(z['X'], z['Y'], sample_weight=W)
        RANKERS.append(m)

def rank_proba(X):
    """Averaged over seeds and class priors -> deterministic and lower variance."""
    return np.mean([m.predict_proba(X)[:, 1] for m in RANKERS], axis=0)

print(f'ranker: {len(RANKERS)} GBMs ({len(CFG.W1_PRIORS)} priors x {len(CFG.SEEDS)} seeds) '
      f'on {z["X"].shape[0]:,} rows x {NFEAT} features')


In [ ]:

# ===================================================================================
#  Run: one ranked list of 25 SMILES per molecule
# ===================================================================================
load_model()
load_pcstore()
pool = build_pool()
print(f'candidate pool: {len(pool.mass):,} structures, {pool.nbits} fingerprint bits', flush=True)

L = load_library(TRAIN)
rep, rep_key, rep_nm = build_rep(L)
print(f'analog reference set: {len(rep):,} spectra (one per structure)', flush=True)

te = pq.read_table(TEST).to_pandas()
te['nm'] = neutral_mass(te.precursor_mz.values.astype(np.float64), te.adduct.values)
mols = list(te.groupby('molecule_id'))
print(f'{len(te):,} spectra / {len(mols):,} molecules to identify', flush=True)

rows, diag = [], []
for gi, (mid, sub) in enumerate(mols):
    nms  = sub.nm.values[np.isfinite(sub.nm.values)]
    smis = []
    if len(nms):
        target = float(np.median(nms))                       # fuse all spectra of the molecule
        specs  = [(r.ms2_mzs, r.ms2_normalized_intensities) for r in sub.itertuples()]

        lib_hits = lib_sim(L, specs, target)                 # Class-1 evidence
        analogs  = analog_sim(L, specs, target, rep, rep_key, rep_nm)   # Class-2 evidence

        cand = pool.window(target, CFG.PPM_WIN)
        if len(cand) == 0:
            cand = pool.window(target, CFG.PPM_FALLBACK)
        if len(cand):
            lv = np.array([lib_hits.get(pool.keys[c], 0.0) for c in cand], np.float32)
            zlog = model_logits(sub)
            if CFG.CAND_CAP and len(cand) > CFG.CAND_CAP:
                # Rank by real evidence, not by mass: library hit first, then the model's f.z.
                coarse = lv * 100.0
                if zlog is not None:
                    mz = pool.fps(cand).astype(np.float32) @ zlog
                    coarse = coarse + (mz - mz.mean()) / max(float(mz.std()), 1e-9)
                else:
                    coarse = coarse - np.abs(pool.mass[cand] - target)
                keep = np.argsort(-coarse)[:CFG.CAND_CAP]
                cand, lv = cand[keep], lv[keep]
            cfp = pool.fps(cand)
            csmi = [pool.smiles[c] for c in cand]
            ex_fp, ex_smi = pubchem_extra(target, zlog, set(csmi))
            if ex_smi:
                cfp = np.concatenate([cfp, ex_fp])
                lv  = np.concatenate([lv, np.zeros(len(ex_smi), np.float32)])
                csmi = csmi + ex_smi
            ids, sims = [], []
            for k, s in analogs:
                i = pool.k2i.get(k, -1)
                if i >= 0: ids.append(i); sims.append(s)
            afp = pool.fps(np.array(ids)) if ids else None
            fsc = frag_scores(csmi, specs,
                              float(np.mean([1.0 if m == 'positive' else -1.0
                                             for m in sub.ionization_mode])))
            X   = rank_features(cfp, lv, afp, np.array(sims, np.float32), zlog, fsc)[:, :NFEAT]
            p   = rank_proba(X)
            order = np.argsort(-p)[:CFG.TOPN]
            smis  = [csmi[i] for i in order]
            diag.append((mid, target, len(csmi), float(lv.max()),
                         float(sims[0]) if sims else 0.0, float(p[order[0]])))
    if not smis: smis = ['CCO']
    rows.append((mid, ';'.join(smis[:CFG.TOPN])))
    if gi % 50 == 0: print(f'  {gi}/{len(mols)}  {time.time()-T0:.0f}s', flush=True)

submission = pd.DataFrame(rows, columns=['molecule_id', 'smiles'])
samp = pd.read_csv(SAMPLE)
submission = samp[['molecule_id']].merge(submission, on='molecule_id', how='left')
submission['smiles'] = submission['smiles'].fillna('CCO')

assert len(submission) == len(samp)
assert submission.molecule_id.duplicated().sum() == 0
assert submission.smiles.isnull().sum() == 0
assert submission.smiles.str.split(';').map(len).max() <= 25
submission.to_csv('submission.csv', index=False)
print(f'\nwrote submission.csv  {submission.shape}   total {time.time()-T0:.0f}s')
submission.head()


In [ ]:

# ===================================================================================
#  Diagnostics — what the engine actually saw
# ===================================================================================
import matplotlib.pyplot as plt
d = pd.DataFrame(diag, columns=['molecule_id','neutral_mass','n_candidates',
                                'best_library_sim','best_analog_sim','top_prob'])
print(d[['n_candidates','best_library_sim','best_analog_sim','top_prob']].describe().round(3).to_string())

fig, ax = plt.subplots(1, 4, figsize=(18, 3.6))
ax[0].hist(d.n_candidates, bins=40, color='#4C72B0'); ax[0].set_title('candidates per molecule'); ax[0].set_xlabel('n')
ax[1].hist(d.best_library_sim, bins=40, color='#DD8452'); ax[1].set_title('best library similarity'); ax[1].set_xlabel('entropy sim')
ax[2].hist(d.best_analog_sim, bins=40, color='#55A868'); ax[2].set_title('best analog similarity'); ax[2].set_xlabel('entropy sim (mass-shifted)')
ax[3].scatter(d.best_library_sim, d.best_analog_sim, s=8, alpha=.5, color='#C44E52')
ax[3].set_xlabel('library sim'); ax[3].set_ylabel('analog sim'); ax[3].set_title('the two evidence channels')
for a in ax: a.spines[['top','right']].set_visible(False)
plt.tight_layout(); plt.show()

# molecules where the library is confident are likely Class 1; the rest lean on analogs
likely_c1 = (d.best_library_sim > 0.85).mean()
print(f'\nmolecules with a confident library hit (sim > 0.85): {likely_c1:.1%}'
      f'  -> the other {1-likely_c1:.1%} are carried by analog propagation')


---

## 🔁 Appendix: how the fingerprint model was trained (so you can beat it)

The weights ship in an attached dataset, but a baseline you cannot retrain is a dead end. The full
training script is below — it is **not executed here** (it needs a GPU and ~2 h); copy it into a
separate notebook with a T4 and the two attached datasets.

**The objective.** Each training spectrum is scored against **63 decoy structures drawn from the
same ±10 ppm mass window** — exactly the competitors it meets at inference — under a softmax
cross-entropy on `f·z`. Because `f·z` *is* the Bayes log-likelihood (see the model section), this
trains the ranking function directly rather than a proxy.

**Three things that mattered, all learned the hard way:**

1. **Save the best-by-validation checkpoint, not the last.** With ~276k distinct structures the
   model memorises fast. Held-out hard-negative top-1 peaks around step 12–20k and then *decays*:

   | step | 9k | 12k | 40k | 69k |
   |---|---|---|---|---|
   | held-out top-1 | 0.446 | **0.457** | ~0.30 | **0.127** |

   Training loss improves the whole way down. My first run shipped the step-69k checkpoint.

2. **Augment.** Peak dropout, intensity jitter and ±5 ppm m/z noise moved the peak from 0.457 to
   0.467 and pushed it out to ~20k steps.

3. **Merge spectra during training — and then feed it merged spectra.** The test gives each molecule
   **1–16 spectra (median 3)**, but the obvious setup trains on single spectra — a real train/test
   mismatch. With `--merge_p 0.6`, 60% of training inputs are 2–4 spectra of the same structure
   collapsed into one peak list (near-duplicate m/z merged, stronger peak kept).

   The catch: **a model only wins on the input distribution it was trained on.**

   | | fed per-spectrum | fed one merged list |
   |---|---|---|
   | trained on single spectra (`fp_single_*`) | **0.4799** | 0.4597 |
   | trained on merged (`fp_merged_*`) | 0.4687 | **0.4896** |

   Feed either one the wrong way and you give up ~0.02 Class-2 MRR. Using each correctly and
   averaging the two gives **0.4904**. Beware also that a merge-trained model's *validation* number
   is computed on merged inputs and so is not comparable to a single-spectrum model's — its 0.511
   hard-negative top-1 looks far better than the single model's 0.453, but on the real Class-2 task
   the gap is much smaller. Always re-check on the task you actually care about.

**Numerical trap worth knowing.** `f·z` with ~800 on-bits and logits of ±5 overflows fp16 and the
softmax saturates. Compute the score in fp32 and centre it within each example —
`sc = (raw − raw.mean(dim=1)) / sqrt(nbits)` — which is a per-example constant shift, so the
ranking is untouched while the magnitudes stay sane. My first run diverged to NaN without it.

**Obvious things to try from here:** pretrain on the unlabelled GeMS/DreaMS corpora the hosts point
to, predict the molecular formula and condition on it, or replace the MetFrag-lite channel with a
real fragmentation-tree scorer (SIRIUS/CFM-ID).


In [ ]:
# Training script - NOT executed here (needs a GPU). Copy into a separate notebook.
# Saved to disk so you can fork this notebook and run it directly.
open("train_fingerprint_model.py","w").write(TRAIN_SCRIPT_SRC)
print("wrote train_fingerprint_model.py", len(TRAIN_SCRIPT_SRC), "bytes")


---

## 🎲 Deciding a bet your validation set cannot score

Pool expansion is the one lever this notebook's holdout is **structurally unable to evaluate**.
Every holdout answer is already in the pool — local recall is **100%** at ±10 ppm — so adding a
database can only ever show up as dilution, never as the recall it buys. Running the experiment and
reading off "it got worse" is not a measurement. It is the shape of the instrument.

So the decision gets made with algebra, and each term gets measured separately.

### The inequality

Let $\rho$ be the pool's true recall on hidden Class-2 molecules, $r(K)$ the chance a
PubChem-supplied answer survives an admission cut of $K$, and $d(K)$ the dilution factor. Expansion
pays exactly when

$$\underbrace{\big[\rho + (1-\rho)\,r(K)\big]}_{\rho'} \cdot d(K) \;>\; \rho
\qquad\Longleftrightarrow\qquad
\rho \;<\; \rho^{*} = \frac{r d}{1 - d + r d}$$

Three of these are measurable without knowing $\rho$.

### Measuring r(K): does the model find the answer among 71 million isomers?

Take a holdout query, **pretend its answer is not in the pool**, pull the PubChem isomers inside the
same ±10 ppm window, and ask where the model's $f\cdot z$ ranks the true structure.

The windows are large — a median of several thousand isomers, up to 30,000. The answer is present in
**72%** of them (stereoisomers collapse onto the same skeleton, so it survives subsampling), and the
model ranks it like this:

| admitted $K$ | 5 | 10 | 20 | 35 | 50 |
|---|---|---|---|---|---|
| $r(K)$ | 0.268 | 0.360 | 0.464 | 0.540 | **0.572** |

Better than it sounds: rank **#1 out of ~1,800 isomers** happened repeatedly. This is the `f·z`
channel doing exactly what CSI:FingerID promises.

### Measuring d(K): what the extra candidates cost

Same holdout, answer left in the pool, so this is pure dilution:

| admitted $K$ | 5 | 10 | 20 | 35 | 50 |
|---|---|---|---|---|---|
| Class-2 $d(K)$ | 0.740 | 0.711 | 0.685 | 0.656 | **0.650** |
| break-even $\rho^{*}$ | 0.433 | 0.470 | 0.503 | 0.507 | **0.515** |

My intuition said a small $K$ would be safer — fewer intruders, less damage. **The arithmetic says
the opposite**: $r$ falls faster than $d$ recovers, so $\rho^{*}$ *rises* with $K$ over this range.
Admitting all ~1,800 isomers is worse again ($d \approx 0.52$), so the optimum is a broad plateau
around $K = 35$–$50$.

**Class 1 is barely touched**: $d_{c_1} = 0.941$ (0.901 → 0.848). A Class-1 molecule has real
library evidence, and the ranker's library features dominate, so PubChem intruders cannot displace
it. The damage is confined to the class with nothing to defend it — the same asymmetry as the
candidate cap, this time with a much gentler slope.

### Two refinements, one of which failed

**Failed: normalising the admission score.** `f·z` is a *sum over set bits*, so a candidate with a
larger fingerprint scores higher for free — the cut might be selecting big molecules rather than
likely ones. Dividing by the bit count (or its square root) should fix that. It does not:

| admission rule | retain@20 | retain@35 | retain@50 | retain@80 |
|---|---|---|---|---|
| **raw `f·z`** | **0.464** | **0.540** | **0.572** | **0.632** |
| `f·z / √bits` | 0.424 | 0.516 | 0.544 | 0.608 |
| `f·z / bits` | 0.364 | 0.460 | 0.532 | 0.592 |

Raw wins at every cut. The bit count is not a nuisance term — a bigger molecule genuinely has more
predicted substructure to match, and `f·z` is the Bayes log-likelihood precisely as written. (Note
that z-scoring within the candidate set is a *monotone* transform and cannot change a ranking, so
bit-count normalisation is the only knob here.)

**Worth doing: widen the window before you sharpen the cut.** `retain(50) = 0.572` factors as

$$\underbrace{0.788}_{\text{answer is in the sampled window}} \times \underbrace{0.79}_{\text{and survives the top-50 cut}}$$

The cut is not the bottleneck — **presence** is, and presence is set purely by how many isomers you
can afford to fingerprint. The windows are large (median 4,772 isomers, p90 16,291, max 36,518):

| isomers fingerprinted per query | 2,000 | 5,000 | 12,000 | all |
|---|---|---|---|---|
| presence | 0.788 | 0.884 | 0.940 | 0.948 |

Raising the cap to 12,000 would lift `retain(50)` to ≈0.74 and the break-even to $\rho^{*} \approx
0.58$, at roughly 6× the fingerprinting cost (about 3 hours of notebook runtime). That is the
cheapest remaining improvement to this lever, and it is a pure engineering budget question rather
than a modelling one.

### The verdict

Folding both classes in, expansion pays overall when $\rho \lesssim 0.48$:

| true pool recall $\rho$ | 0.38 | 0.50 | 0.65 | 0.99 |
|---|---|---|---|---|
| predicted LB | **0.373** | 0.330 | 0.298 | 0.263 |

And the measured bracket for $\rho$ straddles that threshold: **0.377** pooled across the
natural-product libraries, **0.996** on the hosts' own 250-molecule example set. The threshold sits
*inside* the bracket, which is the honest answer — but note both estimates are biased the same way,
**upward**, because every molecule in them has a public reference spectrum and a Class-2 molecule by
definition has none.

So this notebook spends a submission on it. The downside is bounded — a submission that scores badly
does not lower your best — and the upside is the largest single move available. If you have a better
estimate of $\rho$, you can read the answer straight off the table above without running anything.


---

## 📐 How big is Class 2 really? (an open question, sharpened)

A reader worked through my numbers and got a different answer, which is worth addressing head-on
because the disagreement is real and instructive.

What the leaderboard actually pins down is a **product**, not two separate numbers:

$$\text{Class-2 contribution} \;=\; \underbrace{f_2}_{\text{share of test}} \times \underbrace{\rho}_{\text{pool recall}} \times \underbrace{c_2}_{\text{ranking quality}}$$

From the submissions: Class-1 contributes $0.162 \times 0.87 \approx 0.141$, so at LB **0.266** the
Class-2 contribution is $\approx 0.125$, and with $c_2 = 0.539$ that gives

$$f_2 \times \rho \;\approx\; 0.233$$

**That is all the leaderboard tells you.** Splitting it needs an assumption about $\rho$:

* Assume $\rho \approx 0.996$ (COCONUT's coverage of `enveda-np-examples`) → $f_2 \approx 23\%$.
* Assume $\rho \approx 0.5$ → $f_2 \approx 47\%$.

I think the first assumption is optimistic, for a specific reason: **99.6% is measured on compounds
that already have public reference spectra.** A Class-2 molecule is *defined* by having none — it is
exactly the kind of compound a curated natural-product database is least likely to contain. Using
in-library compounds to estimate coverage of out-of-library compounds is the same selection bias
that made my own holdout leak (see the ranker section).

There is also independent evidence from the leaderboard itself. A team at **0.332** with the same
saturated Class-1 ceiling has a Class-2 contribution of $\approx 0.19$. Even granting them a
generous $c_2 = 0.6$ and perfect recall, that needs $f_2 \gtrsim 0.32$; at a realistic $c_2 \approx
0.55$ it needs $f_2 \times \rho \approx 0.35$ — **half again my 0.233**. Since ranking quality
cannot plausibly differ by that much, the likeliest explanation is that **their $\rho$ is higher
than mine**, i.e. a broader candidate database.

So my working estimate is $f_2 \approx 35\text{–}50\%$ with $\rho \approx 0.5\text{–}0.65$ — but I
want to be clear this is an *inference from the leaderboard*, not a measurement.

### Update: a hard lower bound, which settles part of the argument

As the ranker improved, the fitted product rose from $0.233 \to 0.271$. That gives a bound needing
**no assumption about $\rho$ at all**, because $\rho$ is a probability:

$$f_2 \;=\; \frac{f_2\rho}{\rho} \;\ge\; f_2\rho \;=\; 0.271 \qquad (\rho \le 1)$$

**So $f_2 \ge 27\%$, and $f_3 \le 1 - 0.162 - 0.271 \approx 57\%$.** An estimate of $f_2 \approx 18\%$
is not merely unlikely, it is inconsistent with the leaderboard: a test set with 18% Class 2 cannot
produce a Class-2 contribution of 0.271 even with a perfect pool and a perfect ranker.

The bound is also **conservative in the safe direction**. $f_2\rho$ was obtained by dividing the
measured Class-2 contribution by *my* $c_2$, and my holdout is almost certainly easier than the
hidden test's Class-2 slice (see the section on validation not transferring). If the true $c_2$ is
lower than I measure, then $f_2\rho$ is *larger* than 0.271, and the bound only rises.

### The same algebra turns "should I add PubChem?" into one inequality

Adding a database raises recall $\rho \to \rho'$ and dilutes ranking by a factor
$d = c_2^{\text{new}}/c_2$. It is worth doing exactly when

$$\rho' d > \rho.$$

PubChem's dilution is measured here at $d \approx 0.52$ (Class-2 MRR $0.73 \to 0.38$ with the model
channel), and $\rho' \approx 1$ by construction if the hosts define Class 2 by PubChem membership.
So **PubChem pays iff $\rho < 0.52$, i.e. iff $f_2 > 0.271/0.52 \approx 52\%$.**

That is a genuinely close call, and it sits right at the top of my estimated range.

**And here is the uncomfortable part: this notebook cannot measure it.** Every one of my holdout
answers is already in the pool — local recall is **100%** at plus/minus 10 ppm. So a pool expansion
can only ever show me its dilution and never its recall gain. When I reported "PubChem hurts,
measured, do not", that was an honest measurement of *one side of the ledger*, presented as though
it were both. It should read: **PubChem costs 48% of ranking quality, and buys an amount of recall
I have no local instrument to see.**

### Measuring $\rho$: what COCONUT actually covers

So I went and measured it, as far as it can be measured. A Class-2 molecule has no public reference
spectrum, so it cannot be sampled from a spectral library — but the structures *in* the training
libraries come from the same chemistry, and asking how many of them COCONUT contains brackets the
answer:

| library | structures | in COCONUT |
|---|---|---|
| `enveda-np-examples` (the hosts' own example set) | 250 | **99.6%** |
| masaryk | 416 | 83.7% |
| mona | 11,681 | 79.4% |
| msdial | 9,127 | 78.4% |
| riken | 15,892 | 77.2% |
| drug_plus | 2,539 | 75.3% |
| massbank | 9,180 | 61.4% |
| gnps | 45,750 | **36.4%** |
| spectraverse | 9,631 | 21.6% |
| **all NP libraries pooled** | **64,941** | **37.7%** |

Two things jump out.

**The spread is enormous** — 22% to 99.6%. COCONUT's coverage is not a property of natural products,
it is a property of *how curated a particular collection is*. The pooled figure is dragged to 37.7%
by GNPS, which is the largest and least curated of them.

**The 99.6% figure comes from a 250-molecule curated example set**, and every one of those molecules
has public reference spectra — which is exactly the population a natural-product database is *most*
likely to already contain. It is the best-case end of the range, not the typical case. Every number
in that table is biased the same way, upward, because all of them have public spectra and Class-2
molecules by definition do not. So **37.7% is an upper bound on a pooled estimate that is itself an
upper bound.**

Put that next to the decision rule: PubChem pays iff $\rho < 0.52$. The curated end of the range says
no; the pooled end says yes, comfortably. **The honest answer is that this is still open** — but it
is now open between two measured numbers instead of between two assumptions, and the threshold sits
inside the range rather than outside it. If your prior is that the hidden Class-2 molecules look more
like GNPS than like a hand-picked example set — and "no public reference spectrum" is a strong hint
in that direction — then expanding the pool is the highest-value untried move in this competition.

One more consequence worth stating plainly: $f_2 = 0.271/\rho$, so the same table brackets the
Class-2 share at **27% (if $\rho = 1$) to 72% (if $\rho = 0.377$)**. Either way, well above 18%.


---

## What I measured, and what I'd try next

**Things that worked**

* Mass-shifted analog propagation — the single biggest classical lever (0.16 → 0.52 on Class 2).
* A spectrum→fingerprint model ranked by `f·z`, trained on **fused multi-spectrum inputs** —
  **0.517 alone**, level with analog propagation, and **0.63 with all four channels**. Worth +0.033
  on the leaderboard (0.266 → 0.299) the first time it went in, and another +0.036 when it was
  retrained longer.
* Sweeping the ranker's hyperparameters against the leaderboard-calibrated objective rather than
  against validation MRR (+0.007 predicted), and **re-sweeping them whenever a channel improves** —
  the class-prior optimum moved from ~0.50 to 0.45 once the model channel went 0.490 → 0.517, and
  the wide hedge that used to help turned into a 0.008 liability.
* Entropy similarity with entropy weighting instead of plain cosine (Class 1: 0.893 → 0.93).
* A **tight** ±10 ppm candidate window. Tighter is genuinely better — but only at the *window*
  stage, where the filter is mass and mass is what a window is for. Do **not** extend the same
  intuition to a post-window cap (see the candidate-cap section: it costs 24% of Class-2 recall).
* Calibrating the class weight against leaderboard-derived values rather than the raw class share.
* **Bagging the ranker over fixed seeds.** `HistGradientBoostingClassifier` defaults to
  `random_state=None`; two byte-identical notebook versions scored **0.292 and 0.298**. See the
  noise-floor section — this is the single most common way to fool yourself in this competition.

**Measured dead ends — save yourself the time**

| Idea | Result |
|---|---|
| Add **all** PubChem isomers to the pool | 0.52 → 0.35. But this measures only PubChem's *dilution* — local pool recall is already 100%, so the instrument is blind to the recall it buys. Admitting the model's **top 50** instead of all of them changes the arithmetic completely: see *Deciding a bet your validation set cannot score* |
| Soft "consensus fingerprint" averaged over analogs | 0.43 vs 0.52 for max-over-analogs |
| Per-instrument normalisation of analog similarity | 0.517 vs 0.521 |
| Confidence gate to detect "answer not in pool" | AUC 0.627 — too weak to route on |
| Averaging more models **within one input view** | a second merged-input model took the leaderboard from 0.335 → **0.329**. Two models help only when they read the spectra *differently* (per-spectrum vs merged); a second reading of the same kind is redundant |
| Averaging **one** model over two fusion views (per-spectrum *and* merged peak list) | merged-only **0.5163**, per-spectrum 0.4946, mean 0.5087, best weighted blend 0.5134. Even with the scale shared, test-time augmentation loses: feed the model the input distribution it was *trained* on and stop there |
| Re-weighting analogs by how well they fit the predicted fingerprint | 0.521 → 0.512 |
| ChEBI + LIPID MAPS expansion | looked net-positive on validation, leaderboard "disagreed" (0.299 → 0.295) — but that gap is **inside the seed-noise band**, so this one is genuinely *unresolved*, not refuted |
| 3 reference spectra per structure instead of 1 | 0.52 → 0.49 |
| Wider analog window (±400 Da vs ±200) | no change (0.520 vs 0.521) |
| NP-likeness as a prior | 0.037 — worse than random |
| Explicit molecular-formula prediction | only a 1.4× candidate reduction; 72% of window candidates already share the true formula |
| Library similarity as a fixed additive term | Class 2 0.52 → 0.27 |
| Coarse candidate cap at 80 by `lib_sim×100 − \|Δmass\|` (as several forks do) | **−0.055 predicted LB**; drops 24% of Class-2 truths outright. Class-1 retention is 0.992 at every cap, which is why it looks harmless |
| Simple z-score blend of the four channels instead of the GBM | 0.606 vs 0.620 |
| Test-time augmentation: averaging one model over per-spectrum **and** merged views | merged-only 0.516 beats the mean 0.509 and every weighted blend |
| More analogs per molecule | flat above 80: `N_ANALOG` 60 → 0.3683, 80 → 0.3709, 100 → 0.3705, 140 → 0.3703 predicted LB. Nothing here; spend the effort elsewhere |

**Where the remaining score is — and the honest ceiling**

Submissions pin the unknowns in `ExpectedLB = A·c₁ + B·c₂`, where `A = f₁` and `B = f₂ · recall`:

* A library-only submission scores `0.151`, and a deliberately hard cross-library Class-1
  simulation (query with one source library, search with that library **removed**, n=500) gives
  c₁ = 0.872–0.93, with six similarity variants landing within 0.01 of each other. So
  **A = f₁ ≈ 0.162**, and 0.151 is already Class 1's *entire* value. That channel is saturated.
* `B` is then whatever reconciles the equation with the leaderboard. Fitted repeatedly, it has
  drifted **0.20 → 0.233 → 0.271** as the ranker improved. That drift is informative, and it is not
  a constant of nature: it means my Class-2 validation split is systematically *harder* than the
  hidden test's Class-2 slice, so every real improvement lands **larger** on the leaderboard than
  validation predicts. Re-fit `B` after each scored submission; don't treat it as measured.

At the current `A = 0.162`, `B = 0.271`, a leaderboard score of `0.335` implies a hidden-test
c₂ ≈ **0.68** — well above the 0.62 this notebook measures on its own Class-2 holdout, exactly as
the drift predicts.

The ceiling this pool allows is `A + B = 0.162 + 0.271 ≈ **0.43**` — that is what a *perfect*
ranker would score, because `B` already folds in how often the answer is in the pool at all.
So there is ≈0.10 of ranking headroom left and an unknown amount of recall headroom above it.

**What I'd try next, in order of expected value**

1. **A longer-trained / larger fingerprint model.** This channel went 0.47 → 0.517 purely from more
   training, and the 50k-step run peaked at step 36 000 — it is data-limited, not capacity-limited.
   Train on all of `train.parquet` with merge augmentation and it should keep climbing. This is the
   most reliable lever left, and it is the one I ran out of GPU quota on.
2. **Recall, still.** `B = f₂ · recall` caps everything above. The pool needs more Class-2 answers
   *without* near-duplicate decoys — targeted derivative enumeration (glycosylation, hydroxylation,
   methylation of COCONUT scaffolds) rather than all of PubChem.
3. **More ranker seeds.** Bagging is free variance reduction against a 0.0072 noise floor, and
   nobody should be reading 0.004 deltas as signal.
4. **Better Class-1 recall** for molecules whose only reference spectra come from a different
   instrument family.

Ideas, corrections and forks very welcome — the constants are all in `CFG`. 🙂
